In [9]:
# 1. AMBIENTE: VS Code / Jupyter Local
# Centralização de dependências. Importações internas removidas das funções 
# para garantir escopo global e imediata detecção de ausência de bibliotecas.
# ==============================================================================

# !pip install -U "google-genai"
# !pip install python-dotenv networkx matplotlib squarify plotly kaleido
import os
import re
import time
import json
import random
import textwrap
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from google import genai
from google.genai import types
from google.genai import errors
import plotly.express as px
import plotly.graph_objects as go
import requests
import squarify
import sys
import time
import typing_extensions as typing
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import winsound

In [10]:
# 2. Schemas rígidos para forçar a LLM a retornar JSON perfeitamente formatado.
# Nomenclaturas alinhadas hierarquicamente (Código -> Subtema -> Tema).
# ==============================================================================

class EvidenciaExtraida(typing.TypedDict):
    numero_citacao: int
    citacao_direta_verbatim: str       # O trecho cru, sem alterações ou correções gramaticais
    sintese_contextual: str            # Síntese factual sob a máscara sintática obrigatória
    nivel_relevancia: int              # Escala inteira: 1 (Baixa), 2 (Média), 3 (Alta), 4 (Crítica)

class PacoteFamiliarizacao(typing.TypedDict):
    arquivo: str
    chunk_id: int
    evidencias: list[EvidenciaExtraida]

class CodigoGerado(typing.TypedDict):
    numero_citacao: int                  # Elo relacional com a Fase 0
    auditoria_similaridade_semantica: str # Raciocínio CoT comparando a evidência com o Codebook existente
    reutiliza_codigo_preexistente: bool   # Decisão booleana explícita baseada na auditoria semântica
    codigo: str                          # Rótulo restrito pelas travas sintáticas da Opção 1
    justificativa: str                   # Fundamentação analítica focada na engenharia de processos e sistemas de trabalho
    definicao_operacional_curta: str     # Âncora de controle conceitual (1 a 2 frases)

class DescricaoMestre(typing.TypedDict):
    descricao: str                     # Conteúdo analítico detalhado (Essência, Escopo e Relevância)
    definicao_operacional_curta: str    # Âncora de controle resumida (1 a 2 frases)
    nivel_relevancia_meta: int          # Escala inteira de 1 a 4 do impacto processual

class MapeamentoSubtema(typing.TypedDict):
    codigo: str
    subtema_alocado: str
    justificativa_alocacao: str        # NOVO: Fundamentação científica da fusão hierárquica
    definicao_operacional_curta: str   # NOVO: Âncora de controle conceitual (1 a 2 frases) 

class MapeamentoTema(typing.TypedDict):
    subtema: str
    tema_alocado: str
    justificativa_alocacao: str        # NOVO: Justificativa da convergência
    definicao_operacional_curta: str   # NOVO: Âncora de barreira macro (1-2 frases)

class FusaoItem(typing.TypedDict):
    elemento_original: str
    elemento_destino: str              # Nome do elemento sobrevivente ou REUSE se idêntico
    justificativa_fusao: str

class MapeamentoResolvido(typing.TypedDict):
    subtema: str
    tema_final_alocado: str
    justificativa_alocacao: str 

In [11]:
# 3. Configuração de diretórios e definição dos arquivos JSON como 'Fonte Única da Verdade'.
# O estado da aplicação é lido do disco no início de cada arquivo de entrevista e 
# sobrescrito no disco ao final, permitindo injeção de edições manuais do pesquisador.
# ==============================================================================

# CONFIGURAÇÃO DE GATEWAY MULTI-API
PROVEDOR_LLM = "API"            # Opções aceitas: "GEMINI" ou "API"
LLM_API_URL = os.environ.get("LLM_API_URL") # Deve estar declarada no arquivo .env

DIRETORIO_ENTREVISTAS = r"C:\Users\F3807943\Downloads\Análise Temática\Entrevistas"
MODELO_LLM = "gemini-2.5-pro"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
# QTD_VERBATIM_FASE_0 = 2            #DEFINE QUANTAS CITAÇÕES VERBATIM SERÃO EXTRAÍDAS NA FASE 0 PARA CADA CHUNK
MAX_TENTATIVAS = 15
DELAY_API = 5
Matriz_Saturacao_Real = None

DIRETORIO_CHECKPOINTS = os.path.join(DIRETORIO_ENTREVISTAS, "Checkpoints")
DIRETORIO_CACHE_F1 = os.path.join(DIRETORIO_CHECKPOINTS, "Fase1_Chunks")
DIRETORIO_GRAFICOS = os.path.join(DIRETORIO_ENTREVISTAS, "Graficos")
DIRETORIO_RELATORIO = os.path.join(DIRETORIO_ENTREVISTAS, "Relatorio_Final")

os.makedirs(DIRETORIO_CHECKPOINTS, exist_ok=True)
os.makedirs(DIRETORIO_CACHE_F1, exist_ok=True)
os.makedirs(DIRETORIO_GRAFICOS, exist_ok=True)
os.makedirs(DIRETORIO_RELATORIO, exist_ok=True)

ARQUIVO_LOG_PROMPTS = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_prompts.md")

# Arquivos Mestre JSON (Fontes da Verdade Editáveis)
ARQUIVO_CODEBOOK_CODIGOS = os.path.join(DIRETORIO_CHECKPOINTS, "1_codebook_codigos.json")
ARQUIVO_CODEBOOK_SUBTEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "2_codebook_subtemas.json")
ARQUIVO_CODEBOOK_TEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "3_codebook_temas.json")
ARQUIVO_RASTREABILIDADE = os.path.join(DIRETORIO_CHECKPOINTS, "0_rastreabilidade_base.json")
ARQUIVO_HISTORICO_FUSOES = os.path.join(DIRETORIO_CHECKPOINTS, "historico_fagocitose.json")
ARQUIVO_LOG_CONSOLE = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_console.log")

load_dotenv()

# Instâncias Dinâmicas em Memória
Codebook_Codigos = {}
Codebook_Subtemas = {}
Codebook_Temas = {}
Rastreabilidade_Base = []
Metricas_Saturacao = []

Matriz_Saturacao_Antes = None
Matriz_Saturacao_Apos = None

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Variável de ambiente GEMINI_API_KEY não localizada.")
client = genai.Client(api_key=api_key)

In [12]:
# 4. CONTROLE DE ESTADO E INVOCAÇÃO LLM (AUDITORIA AVANÇADA)
# ==============================================================================

def ordenar_arquivos_cronologicamente(diretorio: str) -> list[str]:
    arquivos = [f for f in os.listdir(diretorio) if f.endswith(".txt")]
    def extrair_ordem(nome: str) -> int:
        match = re.search(r"Entrevistado A(\d+)", nome)
        return int(match.group(1)) if match else 9999
    return sorted(arquivos, key=extrair_ordem)

def segmentar_texto(texto: str, chunk_size: int, overlap: int) -> list[str]:
    palavras = texto.split()
    chunks = []
    passo = chunk_size - overlap
    for i in range(0, len(palavras), passo):
        chunk = palavras[i : i + chunk_size]
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(palavras):
            break
    return chunks

def limpar_caracteres_invisiveis(texto: str) -> str:
    if not texto: return ""
    return "".join(c for c in texto if c.isprintable() or c in ['\n', '\t'])


def invocar_llm(prompt: str, schema: typing.Any, instrucao_sistema: str, log_contexto: str = "PROCESSO GLOBAL") -> dict | list:
    time.sleep(DELAY_API)
    
    # =============================================================================
    # HIGH-LEVEL DOCUMENTATION: GERADOR DINÂMICO DE MÁSCARA SINTÁTICA CONDICIONAL
    # Mapeia o tipo do schema de validação para injetar um exemplo de formatação 
    # string diretamente no prompt se o canal de comunicação for a "API" legada.
    # =============================================================================
    if PROVEDOR_LLM == "API":
        exemplo_formato = ""
        
        # Extração segura de tipos estruturais e strings de validação canônica
        schema_str = str(schema)
        
        if schema == PacoteFamiliarizacao:
            exemplo_formato = '{\n    "arquivo": "Nome_Do_Arquivo.txt",\n    "chunk_id": 1,\n    "evidencias": [\n        {\n            "numero_citacao": 1,\n            "citacao_direta_verbatim": "Trecho exato extraído do texto...",\n            "sintese_contextual": "O participante relata que... O contexto envolve... O impacto é...",\n            "nivel_relevancia": 4\n        }\n    ]\n}'
        elif "CodigoGerado" in schema_str:
            exemplo_formato = '[\n    {\n        "numero_citacao": 1,\n        "auditoria_similaridade_semantica": "Raciocínio CoT comparativo...",\n        "reutiliza_codigo_preexistente": false,\n        "codigo": "NOME DO CÓDIGO NOVO",\n        "justificativa": "Fundamentação técnica baseada nos fatos...",\n        "definicao_operacional_curta": "Essência, escopo e relevância do código..."\n    }\n]'
        elif schema == DescricaoMestre:
            exemplo_formato = '{\n    "descricao": "Texto analítico longo contendo Essência, Escopo e Relevância...",\n    "definicao_operacional_curta": "Frase concisa de controle...",\n    "nivel_relevancia_meta": 4\n}'
        elif "MapeamentoSubtema" in schema_str:
            exemplo_formato = '[\n    {\n        "codigo": "NOME DO CÓDIGO",\n        "subtema_alocado": "NOME DO SUBTEMA",\n        "justificativa_alocacao": "Justificativa científica do acoplamento...",\n        "definicao_operacional_curta": "Frase curta ancorando o subtema..."\n    }\n]'
        elif "MapeamentoTema" in schema_str:
            exemplo_formato = '[\n    {\n        "subtema": "NOME DO SUBTEMA",\n        "tema_alocado": "NOME DO TEMA MACRO",\n        "justificativa_alocacao": "Justificativa analítica da convergência...",\n        "definicao_operacional_curta": "Frase curta ancorando o tema macro..."\n    }\n]'
        elif "FusaoItem" in schema_str:
            exemplo_formato = '[\n    {\n        "elemento_original": "NOME ORIGINAL",\n        "elemento_destino": "NOME SOBREVIVENTE OU REUSE",\n        "justificativa_fusao": "Fundamentação qualitativa da fusão..."\n    }\n]'
        elif schema == MapeamentoResolvido:
            exemplo_formato = '{\n    "subtema": "NOME DO SUBTEMA CONFLITANTE",\n    "tema_final_alocado": "TEMA DEFINITIVO ESCOLHIDO",\n    "justificativa_alocacao": "Justificativa conceitual baseada no escopo..."\n}'

        if exemplo_formato:
            bloco_coercao_api = f"""
            
                CRITICAL OUTPUT FORMAT REQUIREMENT:
                Você deve responder obedecendo RIGOROSAMENTE à estrutura e chaves do template JSON abaixo. 
                Não inclua nenhum texto explicativo, notas ou saudações antes ou depois do objeto.

                [TEMPLATE MANDATÓRIO DE SAÍDA]:
                {exemplo_formato}
            """
            instrucao_sistema = f"{instrucao_sistema}\n{bloco_coercao_api}"

    prompt_unificado = (
        f"=== INSTRUÇÕES DE SISTEMA E DIRETRIZES METODOLÓGICAS ===\n{instrucao_sistema}\n\n"
        f"=== CONTEXTO DA ETAPA E DADOS DE ENTRADA ===\n{prompt}"
    )
    
    # ROTEAMENTO DE CLIENTES API
    if PROVEDOR_LLM == "GEMINI":
        resposta = client.models.generate_content(
            model=MODELO_LLM,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=instrucao_sistema,
                temperature=0.0,
                seed=77,
                response_mime_type="application/json",
                response_schema=schema,
                safety_settings=[
                    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
                    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE")
                ]
            )
        )
        texto_retorno = resposta.text.strip()
    
    elif PROVEDOR_LLM == "API":
        payload = {
            'data': {
                'config': {
                    'temperature': 0.0,
                    'max_tokens': 4000
                },
                'input': ' ',
                'contexto': limpar_caracteres_invisiveis(prompt_unificado)
            }
        }
        response = requests.post(LLM_API_URL, json=payload, timeout=90, verify=False)
        response.raise_for_status()
        data = response.json()
        texto_retorno = data['data']['output']['text'][0].strip()
    else:
        raise ValueError(f"Provedor LLM configurado inválido: {PROVEDOR_LLM}")
        
    with open(ARQUIVO_LOG_PROMPTS, "a", encoding="utf-8") as f:
        f.write(f"\n{'#'*80}\nPROVEDOR: {PROVEDOR_LLM} | CONTEXTO: {log_contexto}\n{'#'*80}\n")
        f.write(f"[PROMPT ENVIADO]:\n{prompt if PROVEDOR_LLM == 'GEMINI' else prompt_unificado}\n\n")
        f.write(f"[RETORNO]:\n{texto_retorno}\n\n")
        
    # PARSER EXTRACTOR ROBUSTO COM HIGIENIZADOR SINTÁTICO DE CONTINGÊNCIA
    match_markdown = re.search(r'```json\s*(.*?)\s*```', texto_retorno, re.DOTALL | re.IGNORECASE)
    if match_markdown:
        texto_retorno = match_markdown.group(1)
    else:
        match_brackets = re.search(r'([\\[{].*[\\]}])', texto_retorno, re.DOTALL)
        if match_brackets:
            texto_retorno = match_brackets.group(1)
            
    # Saneamento de segurança ativo para omissão de vírgulas entre chaves/aspas consecutivas
    texto_retorno = re.sub(r'(\s*"(?:[^"\\]|\\.)*")\s*\n\s*(")', r'\1,\n\2', texto_retorno)
            
    return json.loads(texto_retorno)
        
def unificar_descricoes_llm(nome_destino: str, descricoes_originais: list[str], tipo: str) -> str:
    """
    [HIGH-LEVEL DOCUMENTATION]: INTEGRAÇÃO TEÓRICA SIMPLIFICADA (TOKEN OPTIMIZATION)
    Bypass de verbatims e evidências empíricas brutas. Consolida os escopos conceituais 
    exclusivamente com base nas strings textuais das definições mestres concorrentes.
    """
    instrucao = f"""Você é um Especialista Sênior em Engenharia de Processos e Auditoria de Sistemas. 
    Sua tarefa é fundir as descrições conceituais fornecidas para o {tipo} sobrevivente definitivo '{nome_destino}'.
    Combine as Essências, Escopos delimitadores e Relevâncias estratégicas em uma única narrativa fluida, 
    analítica e integrada. Elimine qualquer redundância ou repetição léxica.
    Retorne estritamente um objeto JSON contendo a chave 'descricao' preenchida com o texto unificado."""
    
    prompt = f"DESCRIÇÕES TEXTUAIS ORIGINAIS PARA FUSÃO EM CASCA:\n{json.dumps(descricoes_originais, ensure_ascii=False, indent=4)}"
    
    class UnificacaoSchema(typing.TypedDict):
        descricao: str

    resultado = invocar_llm(prompt, UnificacaoSchema, instrucao, log_contexto=f"OTIMIZAÇÃO TOKEN | FUSÃO DESCRITIVA | {tipo}")
    return resultado.get("descricao", "")

def carregar_estado_json():
    global Codebook_Codigos, Codebook_Subtemas, Codebook_Temas, Rastreabilidade_Base
    if os.path.exists(ARQUIVO_CODEBOOK_CODIGOS):
        with open(ARQUIVO_CODEBOOK_CODIGOS, 'r', encoding='utf-8') as f: Codebook_Codigos.clear(); Codebook_Codigos.update(json.load(f))
    if os.path.exists(ARQUIVO_CODEBOOK_SUBTEMAS):
        with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'r', encoding='utf-8') as f: Codebook_Subtemas.clear(); Codebook_Subtemas.update(json.load(f))
    if os.path.exists(ARQUIVO_CODEBOOK_TEMAS):
        with open(ARQUIVO_CODEBOOK_TEMAS, 'r', encoding='utf-8') as f: Codebook_Temas.clear(); Codebook_Temas.update(json.load(f))
    if os.path.exists(ARQUIVO_RASTREABILIDADE):
        with open(ARQUIVO_RASTREABILIDADE, 'r', encoding='utf-8') as f: Rastreabilidade_Base.clear(); Rastreabilidade_Base.extend(json.load(f))

def salvar_estado_json():
    with open(ARQUIVO_CODEBOOK_CODIGOS, 'w', encoding='utf-8') as f: json.dump(Codebook_Codigos, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'w', encoding='utf-8') as f: json.dump(Codebook_Subtemas, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_CODEBOOK_TEMAS, 'w', encoding='utf-8') as f: json.dump(Codebook_Temas, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_RASTREABILIDADE, 'w', encoding='utf-8') as f: json.dump(Rastreabilidade_Base, f, ensure_ascii=False, indent=4)

In [13]:
# 5. MOTOR METODOLÓGICO: EXTRAÇÃO E ABSTRAÇÃO ESTRUTURADA
# Prompts mantidos estritamente conforme exigido. A lógica injeta dinamicamente o  
# estado existente vs. órfãos na formatação de contexto para propiciar avanço 
# incremental, atuando exclusivamente em objetos sem processamento prévio.
# ==============================================================================

def executar_fase_0_familiarizacao(chunk_texto: str, id_chunk: int, arquivo_nome: str, inicio_contador: int) -> list[dict]:
    # =============================================================================
    # HIGH-LEVEL DOCUMENTATION: MAPER EXTRATIVO COM SEQUENCIAMENTO CRONOLÓGICO
    # Garante a linearidade dos IDs de citações (start=inicio_contador) para evitar
    # colisões de índices numéricos durante a injeção do payload unificado na Fase 1.
    # =============================================================================
    
    # SUBSTITUIR O CORPO DA INSTRUÇÃO DA FASE 0 (instrucao_fase0) POR:
    instrucao_fase0 = """Você é um Engenheiro de Filtros de Extração de Alta Precisão especializado em Auditoria de Sistemas de Trabalho. Sua tarefa é varrer o bloco de texto fornecido e isolar fragmentos literais (verbatims) que evidenciem fricções, barreiras operacionais, dores processuais ou sugestões de automação para o processo de edição, atualização e revisão de normas corporativas (IN).

    DIRETRIZ DE INTEGRALIDADE LINGUÍSTICA (VERBATIM):
    O campo 'citacao_direta_verbatim' DEVE ser uma cópia exata, caractere por caractere, extraída do texto de entrada. Não parafraseie, não resuma, não higienize.

    AVALIAÇÃO PARAMÉTRICA DE RELEVÂNCIA (SEM TETO DE COTA):
    Não há limite máximo de extrações por bloco. Avalie cada evidência isolada e atribua obrigatoriamente um valor inteiro no campo 'nivel_relevancia' seguindo estritamente os critérios:
    1: Relevância Baixa (Comentários periféricos, desabafos sem impacto no processo).
    2: Relevância Média (Dificuldades comuns, desvios operacionais contornáveis, ou do senso comum corporativo).
    3: Relevância Alta (Barreiras que geram retrabalho, latência processual evidente ou duplicação de tarefas).
    4: Relevância Crítica (Paralisia de fluxo, quedas de plataforma, inconsistência de dados oficiais ou risco de conformidade).

    SINTAXE OBRIGATÓRIA DA SÍNTESE CONTEXTUAL (MÁSCARA RÍGIDA):
    O campo 'sintese_contextual' deve seguir estritamente a seguinte estrutura de três períodos factuais, sem qualquer variação verbal:
    1. 'O participante relata que [descrever a fricção ou ação central de forma direta].'
    2. 'O contexto do relato envolve [descrever o cenário, sistema ou processo citado].'
    3. 'O impacto imediato verificado na operação é [descrever a consequência direta descrita].'
    
    EXEMPLO DE REFERÊNCIA (FEW-SHOTS):
    
    Exemplo (Contexto de Suporte TI):
    {
        "arquivo": "Entrevistado_A1.txt",
        "chunk_id": 1,
        "evidencias": [
            {
                "numero_citacao": 1,
                "citacao_direta_verbatim": "o sistema simplesmente travou e eu perdi todo o relatório de atendimento daquela tarde, tive que refazer do zero",
                "sintese_contextual": "O participante relata que perdeu o relatório devido ao travamento da plataforma. O contexto do relato envolve a edição de documentos de atendimento ao cliente. O impacto imediato verificado na operação é a necessidade de retrabalho manual e desperdício de tempo.",
                "nivel_relevancia": 4
            }
        ]
    }
    
    Siga rigorosamente estes padrões e ordene a saída na sequência cronológica exata em que aparecem no texto."""

# MODIFICAR O LOOP DE ESTRUTURAÇÃO DO RESULTADO NA FASE 0 POR:
    prompt_f0 = f"BLOCO DE TEXTO PARA ANÁLISE:\n{chunk_texto}"
    resultado_f0 = invocar_llm(
        prompt=prompt_f0,
        schema=PacoteFamiliarizacao,
        instrucao_sistema=instrucao_fase0,
        log_contexto=f"FASE 0 | Extração Verbatim | Chunk {id_chunk}"
    )
    lista_evidencias = resultado_f0.get("evidencias", [])

    evidencias_estruturadas = []
    for idx, ev in enumerate(lista_evidencias, start=inicio_contador):
        if int(ev.get("nivel_relevancia", 0)) >= 3:
            evidencias_estruturadas.append({
                "Arquivo_Origem": arquivo_nome,
                "ID_Chunk": id_chunk,
                "Numero_Evidencia_Fase0": idx,
                "Trecho_Original": ev.get("citacao_direta_verbatim", "").strip(),
                "sintese_contextual": ev.get("sintese_contextual", "").strip(),
                "Codigo_Atribuido": None,
                "Justificativa_Original": None
            })
    return evidencias_estruturadas

def executar_fase_1(arquivo: str):
    """
    Fase 1: Coleta todas as evidências geradas sequencialmente pela Fase 0 para todo o 
    arquivo da entrevista e executa a codificação conceitual unificada, eliminando chunks nesta etapa.
    Under RIPES Protocol.
    """
    global Codebook_Codigos, Rastreabilidade_Base
    
    caminho_completo = os.path.join(DIRETORIO_ENTREVISTAS, arquivo)
    if not os.path.exists(caminho_completo):
        raise FileNotFoundError(f"[ERRO CRÍTICO RIPES] Arquivo físico de transcrição não localizado: {caminho_completo}")
        
    with open(caminho_completo, "r", encoding="utf-8") as f:
        texto_completo = f.read()
        
    if not texto_completo.strip():
        raise ValueError(f"[ERRO CRÍTICO] O arquivo {arquivo} está completamente vazio no disco.")
    
    # 1. Gera o fatiamento determinístico do corpus (Sliding Window) para saber o total real de blocos
    palavras = texto_completo.split()
    tamanho_chunk = CHUNK_SIZE     
    overlap = CHUNK_OVERLAP         
    
    chunks = []
    i = 0
    while i < len(palavras):
        chunk_palavras = palavras[i : i + tamanho_chunk]
        chunks.append(" ".join(chunk_palavras))
        if i + tamanho_chunk >= len(palavras):
            break
        i += (tamanho_chunk - overlap)
        
    # 2. Controla o ponteiro cronológico sequencial global para novos itens
    if Rastreabilidade_Base:
        contador_global = max(ev["Numero_Evidencia_Fase0"] for ev in Rastreabilidade_Base) + 1
    else:
        contador_global = 1

    # 3. Varre bloco por bloco verificando execução prévia individual
    for id_chunk, chunk_conteudo in enumerate(chunks, start=1):
        
        # Validação cirúrgica: verifica se este bloco exato já foi processado para este arquivo
        chunk_ja_processado = any(
            ev.get("Arquivo_Origem") == arquivo and ev.get("ID_Chunk") == id_chunk 
            for ev in Rastreabilidade_Base
        )
        
        if chunk_ja_processado:
            # Bloco já existe na Source of Truth, passa para o próximo sem gastar tokens
            continue
            
        print(f"    [FASE 0] Extraindo verbatims estruturados inéditos no Bloco {id_chunk}/{len(chunks)}...")
        
        evidencias_chunk = executar_fase_0_familiarizacao(
            chunk_texto=chunk_conteudo,
            id_chunk=id_chunk,
            arquivo_nome=arquivo,
            inicio_contador=contador_global
        )
        
        # Incrementa o ponteiro e anexa os novos resultados encontrados
        contador_global += len(evidencias_chunk)
        Rastreabilidade_Base.extend(evidencias_chunk)
        
        # Gravação imediata no disco para congelar o estado contra interrupções
        salvar_estado_json()

    # 4. Recupera o payload consolidado integral do arquivo (contendo o histórico + os novos blocos gerados)
    todas_evidencias_entrevista = [ev for ev in Rastreabilidade_Base if ev.get("Arquivo_Origem") == arquivo]
    
    payload_filtrado = []
    for ev in todas_evidencias_entrevista:
        payload_filtrado.append({
            "Arquivo_Origem": ev["Arquivo_Origem"],
            "ID_Chunk": ev["ID_Chunk"],
            "Numero_Evidencia_Fase0": ev["Numero_Evidencia_Fase0"],
            "Trecho_Original": ev["Trecho_Original"],
            "sintese_contextual": ev["sintese_contextual"]
        })

    print(f"  [REDUCE - FASE 1] Injetando payload consolidado de {len(payload_filtrado)} verbatims unificados para codificação.")

    instrucao_fase1 = """Você é um Especialista Sênior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho. A sua especialidade é aplicar os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke para decodificar depoimentos operacionais, transformando-os em um Codebook estável e rigorosamente focado na meta desta análise.

    Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

    DEFINIÇÕES CONCEITUAIS CRÍTICAS (BRAUN & CLARKE):
    - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (seja obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica, concisa e direcionada para um fragmento bruto de dados.
    - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
    - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').
    
    DIRETRIZ DE ANCORAGEM EMPÍRICA E VETORES DE CONTEXTO:
    O seu insumo mestre de análise é o campo 'sintese_contextual'. É dele que você deve extrair o significado qualitativo profundo. O campo 'Trecho_Original' serve como adeqaução de linguagem e informação complementar original, para ser usado se necessário.

    TRAVA SINTÁTICA RÍGIDA DE NOMENCLATURA:
    Todo código gerado DEVE ser escrito em texto natural utilizando estritamente espaços simples normais entre as palavras. É expressamente PROIBIDO o uso de underscores (_), hífens, notação snake_case ou a aglutinação/fusão de palavras (ex: NÃO escreva 'EXPOSIÇÃOATENDENTE', escreva 'EXPOSIÇÃO DO ATENDENTE'). O rótulo deve obedecer à seguinte sequência linear separada por espaços comuns: [SUBSTANTIVO ABSTRATO OPERACIONAL] [VETOR DE IMPACTO DIRETO] [CONTEXTO DO FENÔMENO CORPORATIVO]. Proíbe-se terminantemente o uso de verbos conjugados, adjetivos isolados ou frases conversacionais longas que quebrem esta fórmula gramatical.
    
    PROTOCOLO DE VALIDAÇÃO BOOLEANA E CADEIA DE PENSAMENTO:
    Antes de emitir qualquer valor no campo 'codigo', você deve executar uma auditoria de similaridade semântica contra o dicionário do CODEBOOK ATUAL fornecido no prompt. Preencha o campo 'auditoria_similaridade_semantica' explicitando este raciocínio analítico passo a passo (Chain-of-Thought). 
    - Se a nova evidência literal se enquadrar perfeitamente no escopo conceitual de um código preexistente, marque o campo 'reutiliza_codigo_preexistente' como true e repita o nome exato da chave de forma idêntica.
    - Se for um fenômeno inédito à luz da análise de processos de trabalho, marque o campo como false e crie uma nova nomenclatura respeitando rigorosamente a trava sintática.

    DIRETRIZES DE REDAÇÃO, ESCOPO E INTEGRIDADE LINGUÍSTICA:
    - O campo 'definicao_operacional_curta' deve conter uma redação rigorosa, concisa e puramente analítica formatada em texto corrido (2 a 4 frases), cobrindo obrigatoriamente: Essência do fenômeno, Fronteiras de escopo (o que entra e o que sai) e Relevância para o objetivo da pesquisa. 
    - INTEGRALIDADE LINGUÍSTICA DO VERBATIM: Na redação da justificativa e da definição operacional curta, você deve obrigatoriamente reter, incorporar e contextualizar as expressões idiomáticas, gírias corporativas ou erros gramaticais originais presentes no 'Trecho_Original' (ex: 'tiroteio', 'matar no peito', 'colcha de retalhos'). Não higienize nem pasteurize o léxico nativo da linha de frente; utilize estes termos como marcadores de identidade cultural e nuances de significado latente.

    ANÁLISE SISTÊMICA CONSOLIDADA:
    Não processe as evidências de forma isolada, linear ou atômica. Analise o ecossistema completo enviado no payload JSON de uma só vez, capturando os padrões recorrentes e transversais de toda a entrevista e gerando até 15 códigos consolidados por rodada de processamento.
    """
            
    # Normalização Semântica do Contexto Prévio do Codebook
    codebook_contexto = {k: v.get("definicao_operacional_curta", "Sem definição.") for k, v in Codebook_Codigos.items()}
    
    prompt = (
        f"CODEBOOK ATUAL (Estruturas conceituais preexistentes): {json.dumps(codebook_contexto, ensure_ascii=False, sort_keys=True)}\n\n"
        f"CONJUNTO INTEGRAL DE EVIDÊNCIAS EXTRAÍDAS DA FASE 0 PARA CODIFICAÇÃO (ORDEM CRONOLÓGICA):\n{json.dumps(payload_filtrado, ensure_ascii=False, sort_keys=False)}"
    )
       
    resultado_codigos = invocar_llm(
        prompt=prompt, 
        schema=list[CodigoGerado], 
        instrucao_sistema=instrucao_fase1, 
        log_contexto=f"FASE 1 | Codificação Consolidada Verbatim | {arquivo}"
    )
    
    for cd in resultado_codigos:
        num_cit = cd.get("numero_citacao")
        nome_codigo = cd.get("codigo", "").strip().upper()
        
        if not nome_codigo:
            continue
            
        for ev in todas_evidencias_entrevista:
            if ev["Numero_Evidencia_Fase0"] == num_cit:
                ev["Codigo_Atribuido"] = nome_codigo
                ev["Justificativa_Original"] = cd.get("justificativa", "")
                
                if nome_codigo not in Codebook_Codigos:
                    Codebook_Codigos[nome_codigo] = {
                        "descricao": "", 
                        "definicao_operacional_curta": cd.get("definicao_operacional_curta", ""),
                        "subtema_pai": None,
                        "evidencias_brutas": []
                    }
                
                Codebook_Codigos[nome_codigo]["evidencias_brutas"].append({
                    "citacao": ev["Trecho_Original"],
                    "justificativa": cd.get("justificativa", "")
                })

def executar_fase_intermediaria_1():
    
    instrucao = """Você é um Especialista Sénior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho. Sua tarefa é redigir a descrição conceitual mestre para o código fornecido, consolidando suas evidências brutas. Estamos trabalhando com os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke para decodificar depoimentos operacionais, transformando-os em um Codebook estável e rigorosamente focado na meta desta análise.

    DIRETRIZ DE CONSTRUÇÃO CONCEITUAL:
    A descrição deve explicitar: (1) A Essência do código; (2) O Escopo delimitador; (3) A Relevância direta do fenômeno para a Meta de Pesquisa.

    Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

    DEFINIÇÕES CONCEITUAIS CRÍTICAS (BRAUN & CLARKE):
    - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (seja obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica, concisa e direcionada para um fragmento bruto de dados.
    - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
    - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').

    AVALIAÇÃO DE RELEVÂNCIA PROCESSUAL:
    Atribua no campo 'nivel_relevancia_meta' uma nota inteira baseada estritamente no alinhamento do código com a Meta de Pesquisa e na densidade de suas evidências acumuladas:
    1: Periférico (Fricções individuais ou reclamações que não alteram o fluxo normativo).
    2: Médio (Problemas operacionais localizados com baixo impacto na tomada de decisão).
    3: Alto (Gargalos estruturais, retrabalho frequente, latência processual comprovada por múltiplas fontes).
    4: Crítico (Inconsistência de dados oficiais, paralisia de sistemas regulatórios ou falhas graves de conformidade).

    AÇÃO EXIGIDA:
        Com base no pacote de dados brutos recebidos (nome, definição operacional inicial, citações diretas literais e justificativas de campo), consolide a Descrição Mestre definitiva do elemento.
        Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).
    
    DIRETRIZ DE SENSIBILIDADE REFLEXIVA E SIGNIFICAÇÃO LATENTE:
        - Afaste-se da mera descrição superficial do texto. Oriente a síntese para capturar o significado latente: o que está implícito nas entrelinhas das falas, as tensões de poder, os pressupostos organizacionais e as nuances conceituais do fenômeno investigado.
        - Se os relatos apresentarem contradições, paradoxos ou dilemas práticos vividos pelos entrevistados, incorpore essas tensões ativamente no critério de 'Essência' do código,enriquecendo a densidade do estudo.
        - REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado.

    DIRETRIZ DE FAGOCITOSE OPERACIONAL E SANEAMENTO DE SINÔNIMOS:
        - Analise criticamente se os dados empíricos acumulados revelam que este código gerou uma fragmentação excessiva, sobrepondo-se semanticamente a outros conceitos parecidos ou sinônimos do estudo.
        - Force a fusão conceitual: utilize a descrição mestre para expandir o significado deste código de modo que ele absorva e centralize as variações marginais de termos correlatos, unificando a categoria.
        - Use o critério de 'Escopo' para desenhar uma linha divisória clara, explicitando de forma cirúrgica o que este código abrange e o que ele NÃO abrange, blindando o codebook contra novas duplicidades nas próximas iterações.
        
    Retorne a síntese formatada estritamente conforme o schema JSON de descrição.
    
    """
        
    for codigo, dados in Codebook_Codigos.items():
        vistas = set()
        evs_unicas = []
        for d in dados.get("evidencias_brutas", []):
            if d["citacao"] not in vistas:
                vistas.add(d["citacao"])
                evs_unicas.append(d)
        
        amostra = evs_unicas
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")

        prompt = (
            f"NOME DO CÓDIGO: {codigo}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL (Âncora de Controle): {def_inicial}\n"
            f"DADOS EMPÍRICOS ACUMULADOS (Citações Diretas e Justificativas): {json.dumps(amostra, ensure_ascii=False, sort_keys=True)}\n\n"
        )
        
        ctx_log = f"SÍNTESE INT. 1 | Código: {codigo}"
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        
        # FIX DE PERSISTÊNCIA INTEGRAL DE METADADOS
        Codebook_Codigos[codigo]["descricao"] = resultado["descricao"]
        if resultado.get("definicao_operacional_curta"):
            Codebook_Codigos[codigo]["definicao_operacional_curta"] = resultado["definicao_operacional_curta"]
        Codebook_Codigos[codigo]["nivel_relevancia_meta"] = resultado["nivel_relevancia_meta"]
        print(f"  [SÍNTESE CÓDIGO] {codigo} -> Escrito com Relevância Meta {resultado['nivel_relevancia_meta']}.")

def executar_fase_2():
    
    instrucao = """
        Você atua estritamente como um Especialista Sénior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho, utilizando os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke. O seu método hermenêutico rejeita inferências superficiais e categorizações ad-hoc de senso comum corporativo. A sua capacidade analítica deve guiar-se estritamente pela identificação, categorização e estruturação de gargalos operacionais, fluxos informacionais latentes e requisitos conceituais de design para automação.
        Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

        DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):     
        - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação. Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.
        - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
        - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador.
                 
        DIRETRIZ DE USO DE CITAÇÕES EMPÍRICAS (SUPORTE CONTEXTUAL):
        Os trechos enviados no campo 'citacoes_empiricas' são amostras exclusivas para calibração de linguagem enriquecimento do seu vocabulário operacional e complementação original para compreensão do dialeto corporativo real. Use-a somente como âncora linguística e contextual para entender as conexões lógicas entre as categorias, caso necessário.
                         
        TRAVA SINTÁTICA RÍGIDA:
        O nome do Subtema deve seguir estritamente o padrão sintático: [SUBSTANTIVO ABSTRATO OPERACIONAL] + [CONTEXTO DE GOVERNANÇA]. Exemplo: GOVERNANÇA DE PROCESSOS, ARQUITETURA DE DADOS EMERGENTE, ADOÇÃO DE INTELIGÊNCIA ARTIFICIAL.

        REGRAS DE FORMATAÇÃO E AGRUPAMENTO:
        - Analise o código órfão em relação aos SUBTEMAS EXISTENTES informados acima. Se o conceito for um sinônimo direto ou variação semântica de uma categoria já estabelecida, REUTILIZE o nome exato da chave.
        - A matriz resultante DEVE ser limitada a no MÁXIMO 20 subtemas lógicos no total de todo o projeto.
        - REGRA ABSOLUTA: O nome do subtema deve ter MÁXIMA SIMPLICIDADE (1 a 3 palavras no máximo).
        - REGRA LINGUÍSTICA: Mantenha o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado.

        AÇÃO EXIGIDA:
        - Analise criticamente o código órfão em relação aos SUBTEMAS EXISTENTES informados acima. Se o conceito for um sinônimo direto ou uma variação semântica de uma categoria já estabelecida, REUTILIZE o nome exato da chave de forma idêntica para evitar pulverização.
        - CONTUDO, se o código revelar uma faceta complementar, uma tensão prática, uma barreira específica ou uma nuance que expande o fenômeno para além das fronteiras dos subtemas atuais, VOCÊ DEVE CRIAR UM NOVO SUBTEMA.
        - Não force o agrupamento de conceitos distintos em 'conjuntos genéricos' apenas para economizar categorias. A riqueza analítica e a multiplicidade do campo empírico devem ser preservadas.
        - A matriz resultante DEVE ser limitada a no MÁXIMO 20 subtemas lógicos no total de todo o projeto.
        - Para cada alocação, preencha rigorosamente a 'justificativa_alocacao'. Caso um subtema INÉDITO seja criado, forneça uma 'definicao_operacional_curta' de controle (máximo 2 frases).
        Retorne o mapeamento relacional estritamente conforme o schema JSON solicitado.
        """              
               
    # Isola códigos órfãos e monta payload rico com contexto empírico
    codigos_alocados = set()
    for dados_subtema in Codebook_Subtemas.values():
        codigos_alocados.update(dados_subtema.get("codigos_vinculados", []))
        
    vistas_codigos = set()
    codigos_orfaos = []
    for k, v in Codebook_Codigos.items():
        if k not in codigos_alocados:
            vistas_citacoes = set()
            evs_unicas = []
            for d in v.get("evidencias_brutas", []):
                if d["citacao"] not in vistas_citacoes:
                    vistas_citacoes.add(d["citacao"])
                    evs_unicas.append(d)

            amostra_estabilizada = evs_unicas[:5]

            codigos_orfaos.append({
            "codigo": k,
            "definicao_teorica": v.get("descricao", ""),
            "citacoes_empiricas": [e["citacao"] for e in amostra_estabilizada]
            })
    
    if not codigos_orfaos:
        print("  [FASE 2] Nenhum código órfão para mapeamento.")
        return
    
    # HIGH-LEVEL DOCUMENTATION: GARANTIA DE DETERMINISMO VIA ORDENAÇÃO ALFABÉTICA
    # Força a lista de códigos órfãos a manter a mesma sequência exata em todas as execuções.
    codigos_orfaos = sorted(codigos_orfaos, key=lambda x: x["codigo"])

    # ANCORAGEM SEMÂNTICA: Extração dos subtemas existentes acompanhados de seus escopos teóricos
    subtemas_contexto = {}
    for k, v in Codebook_Subtemas.items():
        subtemas_contexto[k] = v.get("descricao", "Sem descrição definida até o momento.")

    # Ativação de Semente Determinística via serialização ordenada do payload de órfãos
    prompt = (
        f"CONTEXTO GLOBAL - SUBTEMAS EXISTENTES E SEUS ESCOPOS: {json.dumps(subtemas_contexto, ensure_ascii=False, sort_keys=True)}\n\n"
        f"LISTA DE CÓDIGOS ÓRFÃOS PARA ANÁLISE:\n{json.dumps(codigos_orfaos, ensure_ascii=False, sort_keys=True)}\n\n"
    )
          
    ctx_log = f"FASE 2 | Mapeamento de {len(codigos_orfaos)} códigos órfãos"
    resultado = invocar_llm(prompt, list[MapeamentoSubtema], instrucao, log_contexto=ctx_log)
    
    for item in resultado:
        cod_nome = item["codigo"].upper().strip()
        subtema_nome = item["subtema_alocado"].upper().strip()
        
        # INTERCEPTAÇÃO E RESOLUÇÃO AUTOMÁTICA DE CONFLITO HOMÔNIMO (Nível 3 para Nível 2)
        if subtema_nome == cod_nome or subtema_nome in Codebook_Codigos:
            subtema_nome += "_"
            
        if subtema_nome not in Codebook_Subtemas:
            Codebook_Subtemas[subtema_nome] = {
                "descricao": "", 
                "definicao_operacional_curta": item.get("definicao_operacional_curta", "").strip(),
                "codigos_vinculados": [], 
                "justificativas_agrupamento": [],
                "precisa_revisao": True
            }
            
        if cod_nome not in Codebook_Subtemas[subtema_nome]["codigos_vinculados"]:
            Codebook_Subtemas[subtema_nome]["codigos_vinculados"].append(cod_nome)
            Codebook_Subtemas[subtema_nome]["justificativas_agrupamento"].append({
                "codigo": cod_nome,
                "justificativa": item.get("justificativa_alocacao", "")
            })
            Codebook_Subtemas[subtema_nome]["precisa_revisao"] = True
            
        print(f"  [MAP] {cod_nome} -> {subtema_nome}")

def executar_fase_intermediaria_2():
    instrucao = """
        Você atua estritamente como um Especialista Sénior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho, utilizando os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke. O seu método hermenêutico rejeita inferências superficiais e categorizações ad-hoc de senso comum corporativo. A sua capacidade analítica deve guiar-se estritamente pela identificação, categorização e estruturação de gargalos operacionais, fluxos informacionais latentes e requisitos conceituais de design para automação.
        Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

        DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):     
        - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação. Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.
        - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
        - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador.

        DIRETRIZ DE USO DE CITAÇÕES EMPÍRICAS (SUPORTE CONTEXTUAL):
        Os trechos enviados no campo 'EVIDÊNCIAS EMPÍRICAS INTEGRAIS' servem estritamente como suporte para enriquecimento do vocabulário operacional e ancoragem do contexto dialetal real. Use-os como guias de calibração semântica, mantendo sua capacidade de abstração analítica.

        DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:
        O JSON de retorno deve conter obrigatoriamente três campos mapeados:
        1. 'descricao': Texto corrido explicitando a Essência do padrão identificado, as Fronteiras de Escopo (o que entra e o que sai) e a Relevância para o Objetivo da Pesquisa.
        2. 'definicao_operacional_curta': Uma frase concisa ancorando o conceito.
        3. 'nivel_relevancia_meta': Nota inteira de 1 a 4 baseada no impacto do subtema sobre o fluxo de trabalho.
        
        AÇÃO EXIGIDA:
            Com base no pacote analítico completo acima, gere ou refine a Descrição Mestre definitiva deste Subtema (Categoria).
            Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).
            
        DIRETRIZ DE CONSTRUÇÃO CONCEITUAL INTERPRETATIVA:
            - Desenvolva a descrição mestre tratando este subtema como uma estrutura interpretativa latente. Explique o padrão conceitual subjacente que une esses códigos, detalhando as implicações psicológicas, sociais ou de gestão que emergem das evidências.
            - Garanta que a descrição não seja uma simples lista ou colagem dos códigos filhos, mas uma narrativa teórica integrada, fluida e analítica sobre uma faceta específica da pergunta de pesquisa.
            
        DIRETRIZ DE RIGIDEZ DE ESCOPO E ISOLAMENTO CATEGÓRICO:
        - É terminantemente proibido fundir ou absorver subtemas que representem facetas complementares, dimensões diferentes ou tensões distintas do mesmo fenômeno nesta etapa.
        - Concentre-se exclusivamente em descrever o conteúdo intrínseco dos códigos filhos vinculados. Utilize o critério de 'Escopo' para refinar as fronteiras lógicas desta categoria, garantindo que ela mantenha sua identidade analítica individualizada sem invadir ou mesclar-se com categorias vizinhas correlatas.

        REGRA LINGUÍSTICA ABSOLUTA: Incorpore as expressões vernacular-corporativas originais presentes nas citações (ex: 'tiroteio', 'matar no peito', 'colcha de retalhos') como marcadores de nuances de significado do ecossistema de trabalho.
            
        Retorne a síntese formatada estritamente conforme o schema JSON de descrição.
    """

    
    for subtema, dados in list(Codebook_Subtemas.items()):
        evidencias = []
        for cod_nome in dados.get("codigos_vinculados", []):
            if cod_nome in Codebook_Codigos:
                evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))
        
        vistas_citacoes = set()
        evs_unicas = []
        for d in evidencias:
            if d["citacao"] not in vistas_citacoes:
                vistas_citacoes.add(d["citacao"])
                evs_unicas.append(d)
        
        amostra = evs_unicas
        descricao_anterior = dados.get("descricao", "")
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")
        justificativas_historicas = dados.get("justificativas_agrupamento", [])
        ctx_log = f"SÍNTESE INT. 2 | Subtema: {subtema}"

        prompt = (
            f"NOME DA CATEGORIA (SUBTEMA): {subtema}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL: {def_inicial}\n"
            f"JUSTIFICATIVAS DE MAPEAMENTO DOS CÓDIGOS PAI: {json.dumps(justificativas_historicas, ensure_ascii=False)}\n"
            f"SUBTEMAS INCORPORADOS NESTA RODADA (FAGOCITOSE): {json.dumps(dados.get('historico_absorcoes', []), ensure_ascii=False)}\n" 
            f"DESCRIÇÃO CONCEITUAL ACUMULADA: {descricao_anterior if descricao_anterior else 'Inexistente. Construção do escopo inicial.'}\n"
            f"EVIDÊNCIAS EMPÍRICAS INTEGRAIS (Citações e Justificativas de Códigos): {json.dumps(amostra, ensure_ascii=False)}\n\n"
        )
                                                
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        
        # FIX DE PERSISTÊNCIA INTEGRAL DE METADADOS
        Codebook_Subtemas[subtema]["descricao"] = resultado["descricao"]
        Codebook_Subtemas[subtema]["definicao_operacional_curta"] = resultado["definicao_operacional_curta"]
        Codebook_Subtemas[subtema]["nivel_relevancia_meta"] = resultado["nivel_relevancia_meta"]
        Codebook_Subtemas[subtema]["precisa_revisao"] = False
        print(f"  [SÍNTESE SUBTEMA] {subtema} -> Atualizado/Escrito.")

def executar_fase_3():

    instrucao = """
        Você atua estritamente como um Especialista Sénior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho, utilizando os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke. O seu método hermenêutico rejeita inferências superficiais e categorizações ad-hoc de senso comum corporativo.
        Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

        DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):
        - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.
        - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
        - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').

        TRAVA SINTÁTICA RÍGIDA (OPÇÃO 1):
        O nome do Tema deve seguir o padrão: [SUBSTANTIVO CATEGÓRICO] + [QUALIFICADOR SISTÊMICO]. Máximo de 1 a 3 palavras. Exemplos: ECOSSISTEMA INFORMACIONAL, ESTRUTURA E GOVERNANÇA, SIMBIOSE COGNITIVA.

        AÇÃO EXIGIDA:
        Abaixo está a lista de novos subtemas órfãos. Para cada subtema, é fornecida sua 'definicao_teorica' e uma amostra de 'citacoes_empiricas' basais.
        1. Utilize a definição e as citações para auditar a abrangência real do subtema e elevá-lo ao nível macro de abstração (Temas).

        DIRETRIZ DE USO DE CITAÇÕES EMPÍRICAS (SUPORTE CONTEXTUAL):
        Os trechos enviados no campo 'citacoes_empiricas' são amostras exclusivas para calibração de linguagem enriquecimento do seu vocabulário operacional e complementação original para compreensão do dialeto corporativo real. Use-a somente como âncora linguística e contextual para entender as conexões lógicas entre as categorias, caso necessário.

        DIRETRIZ DE INTEGRALIDADE TAXONÔMICA E AUTONOMIA (REPLICABILIDADE CIENTÍFICA):
        - Analise o subtema órfão em relação aos TEMAS EXISTENTES. Se o conceito se alinhar, complementar ou expandir o núcleo de significado de um tema já criado, REUTILIZE o nome exato da chave correspondente.
        - O limite máximo absoluto permanece em 10 temas globais para todo o projeto corporativo, forçando a síntese apenas quando houver convergência real de significado.
        - Para cada alocação, preencha a 'justificativa_alocacao' demonstrando o nexo causal. Caso crie um tema INÉDITO, forneça sua 'definicao_operacional_curta' de controle (1-2 frases).
        - REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado, capturando o real significado latente do fenômeno corporativo investigado.

        Retorne o mapeamento relacional estritamente conforme o schema JSON solicitado.
    """
            
    subtemas_alocados = set()
    for dados in Codebook_Temas.values(): subtemas_alocados.update(dados.get("subtemas_vinculados", []))
    
    subtemas_orfaos = []
    for k, v in Codebook_Subtemas.items():
        if k not in subtemas_alocados:
            evidencias = []
            for cod_nome in v.get("codigos_vinculados", []):
                if cod_nome in Codebook_Codigos:
                    evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))
            
            vistas_citacoes = set()
            evs_unicas = []
            for d in evidencias:
                if d["citacao"] not in vistas_citacoes:
                    vistas_citacoes.add(d["citacao"])
                    evs_unicas.append(d)
                    
            amostra_estabilizada = evs_unicas[:10] 
            
            subtemas_orfaos.append({
                "subtema": k,
                "definicao_teorica": v.get("descricao", ""),
                "citacoes_empiricas": [e["citacao"] for e in amostra_estabilizada]
            })
    
    if not subtemas_orfaos:
        print("  [FASE 3] Nenhum subtema órfão para mapeamento.")
        return
    
    # HIGH-LEVEL DOCUMENTATION: NEUTRALIZAÇÃO DE ASSINCRONIA DE MEMÓRIA
    # Ordenação alfabética estrita para congelar a estrutura do payload de entrada da Fase 3.
    subtemas_orfaos = sorted(subtemas_orfaos, key=lambda x: x["subtema"])

    # ANCORAGEM SEMÂNTICA: Mapeia Temas existentes para suas definições conceituais (Remove a lista de strings plana)
    temas_contexto = {}
    for k, v in Codebook_Temas.items():
        temas_contexto[k] = v.get("descricao", "Sem descrição definida até o momento.")

    prompt = (
        f"CONTEXTO GLOBAL - TEMAS EXISTENTES E SEUS ESCOPOS: {json.dumps(temas_contexto, ensure_ascii=False)}\n\n"
        f"SUBTEMAS ÓRFÃOS PARA ANÁLISE:\n{json.dumps(subtemas_orfaos, ensure_ascii=False)}\n\n"
    )
                
    ctx_log = f"FASE 3 | Mapeamento de {len(subtemas_orfaos)} subtemas órfãos"
    resultado = invocar_llm(prompt, list[MapeamentoTema], instrucao, log_contexto=ctx_log)
    
    for item in resultado:
        subtema_nome = item["subtema"].upper().strip()
        tema_nome = item["tema_alocado"].upper().strip()
        
        # INTERCEPTAÇÃO E RESOLUÇÃO AUTOMÁTICA DE CONFLITO HOMÔNIMO TRANS-HIERÁRQUICO (Nível 2 para Nível 1)
        if tema_nome == subtema_nome or tema_nome in Codebook_Subtemas or tema_nome in Codebook_Codigos:
            tema_nome += "__"
            
        if tema_nome not in Codebook_Temas:
            Codebook_Temas[tema_nome] = {
                "descricao": "", 
                "definicao_operacional_curta": item.get("definicao_operacional_curta", "").strip(),
                "subtemas_vinculados": [], 
                "justificativas_agrupamento": [],
                "precisa_revisao": True
            }
            
        if subtema_nome not in Codebook_Temas[tema_nome]["subtemas_vinculados"]:
            Codebook_Temas[tema_nome]["subtemas_vinculados"].append(subtema_nome)
            Codebook_Temas[tema_nome]["justificativas_agrupamento"].append({
                "subtema": subtema_nome,
                "justificativa": item.get("justificativa_alocacao", "")
            })
            Codebook_Temas[tema_nome]["precisa_revisao"] = True
            
        print(f"  [MAP] {subtema_nome} -> {tema_nome}")

def executar_fase_intermediaria_3():
    # SUBSTITUIR A INICIALIZAÇÃO DA VARIÁVEL instrucao E O LAÇO FOR NA FUNÇÃO executar_fase_intermediaria_3 POR:
    instrucao = """
        Você atua estritamente como um Especialista Sénior em Engenharia de Processos e Análise Qualitativa de Sistemas de Trabalho, utilizando os fundamentos da vertente de Codebook Thematic Analysis (Codebook TA) de Braun e Clarke. O seu método hermenêutico rejeita inferências superficiais e categorizações ad-hoc de senso comum corporativo. A sua capacidade analítica deve guiar-se estritamente pela identificação, categorização e estruturação de gargalos operacionais, fluxos informacionais latentes e requisitos conceituais de design para automação.
        Meta desta Análise: Mapear o processo complexo de edição, revisão e atualização de Instruções Normativas (INs) para identificar barreiras operacionais e entender como a Automação Inteligente pode ser implementada para aumentar as capacidades cognitivas e apoiar a tomada de decisão.

        DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):     
        - CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação. Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.
        - SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.
        - TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador.

        DIRETRIZ DE USO DE CITAÇÕES EMPÍRICAS (SUPORTE CONTEXTUAL):
        As citações empíricas integradas servem exclusivamente para suporte de vocabulário e refinamento de contexto. Baseie-se nelas para ancorar o dialeto real do sistema de trabalho, focando na macroestrutura do fenômeno.

        DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:
        O JSON de retorno deve preencher rigorosamente os campos:
        1. 'descricao': Narrativa integrada detalhando a Essência do macro-padrão, o Escopo delimitador e a Relevância para a Meta de Pesquisa.
        2. 'definicao_operacional_curta': Uma frase concisa de controle.
        3. 'nivel_relevancia_meta': Nota inteira de 1 a 4 baseada na criticidade do tema para o processo decisório.

        AÇÃO EXIGIDA:
            Com base no pacote epistêmico completo fornecido, gere ou refine a Descrição Mestre teórica definitiva deste Tema Estruturante.
            Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).

        DIRETRIZ DE RIGIDEZ DE ESCOPO E BLINDAGEM CONCEITUAL:
        - Esta é a camada de maior nível de abstração da pesquisa. Concentre-se estritamente em delimitar a Essência e o Escopo específico do tema atual com base unicamente nos subtemas a ele vinculados.
        - É terminantemente proibido tentar resolver sobreposições horizontais com temas vizinhos ou antecipar fusões textuais nesta etapa. Use o critério de 'Escopo' para desenhar uma linha divisória inflexível, deixando claro o que este tema abrange e o que ele NÃO abrange, mantendo sua identidade taxonômica limpa e isolada para que o processo de harmonização global pós-corpus ocorra sem contaminação prévia.

        REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado, capturando o real significado latente do fenômeno corporativo investigado.
            
        Retorne a síntese formatada estritamente conforme o schema JSON de descrição.
    """   
    
    for tema, dados in list(Codebook_Temas.items()):
        evidencias = []
        for sub_nome in dados.get("subtemas_vinculados", []):
            if sub_nome in Codebook_Subtemas:
                for cod_nome in Codebook_Subtemas[sub_nome].get("codigos_vinculados", []):
                    if cod_nome in Codebook_Codigos:
                        evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))
        
        vistas_citacoes = set()
        evs_unicas = []
        for d in evidencias:
            if d["citacao"] not in vistas_citacoes:
                vistas_citacoes.add(d["citacao"])
                evs_unicas.append(d)
        
        amostra = evs_unicas
        descricao_anterior = dados.get("descricao", "")
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")
        justificativas_historicas = dados.get("justificativas_agrupamento", [])
        
        prompt = (
            f"NOME DO TEMA ESTRUTURANTE: {tema}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL: {def_inicial}\n"
            f"JUSTIFICATIVAS DE ACOPLAMENTO DOS SUBTEMAS FILHOS: {json.dumps(justificativas_historicas, ensure_ascii=False)}\n"
            f"DESCRIÇÃO CONCEITUAL ACUMULADA DO TEMA: {descricao_anterior if descricao_anterior else 'Inexistente. Construção inicial do macro-padrão.'}\n"
            f"EVIDÊNCIAS EMPÍRICAS CONSOLIDADAS (Insumos dos Códigos e Subtemas): {json.dumps(amostra, ensure_ascii=False)}\n\n"
        )
    
        ctx_log = f"SÍNTESE INT. 3 | Tema: {tema}"
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        
        # FIX DE PERSISTÊNCIA INTEGRAL DE METADADOS
        Codebook_Temas[tema]["descricao"] = resultado["descricao"]
        Codebook_Temas[tema]["definicao_operacional_curta"] = resultado["definicao_operacional_curta"]
        Codebook_Temas[tema]["nivel_relevancia_meta"] = resultado["nivel_relevancia_meta"]
        Codebook_Temas[tema]["precisa_revisao"] = False
        print(f"  [SÍNTESE TEMA] {tema} -> Atualizado/Escrito.")

def registrar_e_executar_fagocitose(tipo_fase: str, chave_destinataria: str, chaves_fagocitadas: list[str]):
    """
    Sincroniza a memória física dos dicionários e o arquivo de auditoria 
    sempre que a LLM decreta uma fusão de categorias em tempo de execução.
    """
    if not chaves_fagocitadas:
        return
        
    historico = {}
    if os.path.exists(ARQUIVO_HISTORICO_FUSOES):
        with open(ARQUIVO_HISTORICO_FUSOES, 'r', encoding='utf-8') as f:
            historico = json.load(f)
            
    for chave_morta in chaves_fagocitadas:
        chave_morta_clean = chave_morta.upper().strip()
        if chave_morta_clean == chave_destinataria or not chave_morta_clean:
            continue
            
        # Executa a fusão física na memória do pipeline
        if tipo_fase == "SUBTEMA" and chave_morta_clean in Codebook_Subtemas:
            # Transfere os códigos vinculados para o subtema sobrevivente
            for cod in Codebook_Subtemas[chave_morta_clean].get("codigos_vinculados", []):
                if cod not in Codebook_Subtemas[chave_destinataria]["codigos_vinculados"]:
                    Codebook_Subtemas[chave_destinataria]["codigos_vinculados"].append(cod)
            
            # NOVA INJEÇÃO: Preserva a descrição do subtema morto antes de deletá-lo
            if "historico_absorcoes" not in Codebook_Subtemas[chave_destinataria]:
                Codebook_Subtemas[chave_destinataria]["historico_absorcoes"] = []
            Codebook_Subtemas[chave_destinataria]["historico_absorcoes"].append({
                "subtema_absorvido": chave_morta_clean,
                "descricao_anterior": Codebook_Subtemas[chave_morta_clean].get("descricao", "")
            })
            
            # Remove a chave duplicada
            del Codebook_Subtemas[chave_morta_clean]
            
        elif tipo_fase == "TEMA" and chave_morta_clean in Codebook_Temas:
            # Transfere os subtemas vinculados para o tema sobrevivente
            for sub in Codebook_Temas[chave_morta_clean].get("subtemas_vinculados", []):
                if sub not in Codebook_Temas[chave_destinataria]["subtemas_vinculados"]:
                    Codebook_Temas[chave_destinataria]["subtemas_vinculados"].append(sub)
            del Codebook_Temas[chave_morta_clean]
            
        # Registra a mutação no arquivo JSON de auditoria histórica
        log_id = f"{tipo_fase}_{time.strftime('%Y%m%d_%H%M%S')}"
        historico[log_id] = {
            "Tipo": tipo_fase,
            "Chave_Absorvida": chave_morta_clean,
            "Chave_Destino_Consolidada": chave_destinataria,
            "Data_Hora": time.strftime('%Y-%m-%d %H:%M:%S')
        }
        print(f"  [FAGOCITOSE OPERADA] {chave_morta_clean} foi totalmente colapsada em -> {chave_destinataria}")
        
    with open(ARQUIVO_HISTORICO_FUSOES, 'w', encoding='utf-8') as f:
        json.dump(historico, f, ensure_ascii=False, indent=4)
    
def sanear_hierarquia_quoruns():
    """
    [HIGH-LEVEL DOCUMENTATION]: AUDITORIA DE QUÓRUM E SANEAMENTO TAXONÔMICO PASSIVO
    Varre as estruturas hierárquicas para identificar subtemas órfãos ou temas
    sem quórum mínimo. Bloqueia a alteração estrutural automática em favor da 
    intervenção humana por meio de flags de controle e emissão de relatório físico Markdown.
    """
    global Codebook_Temas, Codebook_Subtemas
    print("\n  [SANEAMENTO] Iniciando varredura passiva de quórum estrutural nos Temas...")
    
    alertas = []
    
    # 1. Auditoria de Subtemas vinculados a Temas (Nível 1 -> Nível 2)
    for tema, dados in Codebook_Temas.items():
        subs = dados.get("subtemas_vinculados", [])
        if len(subs) < 2:
            dados["pendente_revisao_humana"] = True
            msg = f"Tema '{tema}' possui quórum insuficiente ({len(subs)} subtemas vinculados: {subs})."
            print(f"    [ALERTA HIERÁRQUICO] {msg}")
            alertas.append(f"- **Falta de Quórum em Tema**: {msg}")
            
    # 2. Auditoria de Códigos vinculados a Subtemas (Nível 2 -> Nível 3)
    for subtema, dados in Codebook_Subtemas.items():
        codigos = dados.get("codigos_vinculados", [])
        if len(codigos) < 2:
            dados["pendente_revisao_humana"] = True
            msg = f"Subtema '{subtema}' possui densidade insuficiente ({len(codigos)} códigos vinculados: {codigos})."
            print(f"    [ALERTA HIERÁRQUICO] {msg}")
            alertas.append(f"- **Baixa Densidade em Subtema**: {msg}")
            
    # Criação do relatório físico de trilha de auditoria para intervenção humana
    caminho_relatorio = os.path.join(DIRETORIO_RELATORIO, "relatorio_alertas_saneamento.md")
    with open(caminho_relatorio, "w", encoding="utf-8") as f:
        f.write("# RELATÓRIO DE AUDITORIA E ALERTAS DE SANEAMENTO HIERÁRQUICO\n\n")
        f.write(f"Data de Geração: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        if alertas:
            f.write("Foram identificadas inconsistências estruturais que demandam intervenção analítica humana:\n\n")
            f.write("\n".join(alertas))
        else:
            f.write("Nenhuma inconsistência de quórum ou subquórum foi localizada no ecossistema.\n")
    print(f"  [SANEAMENTO PASSIVO] Relatório de alertas exportado em: {caminho_relatorio}")

def exportar_artefatos(sufixo_arquivo="", dados_evolucao: dict = None):
    """
    [HIGH-LEVEL DOCUMENTATION]: GERADOR ESTRUTURAL DE ARTEFATOS EM EXCEL
    Suporta a gravação de abas distintas para a curva de saturação teórica 
    dupla (Matriz_Saturacao_Antes e Matriz_Saturacao_Apos) no snapshot consolidado final.
    """
    global Matriz_Saturacao_Antes, Matriz_Saturacao_Apos
    df_rastreabilidade = pd.DataFrame(Rastreabilidade_Base)
    if df_rastreabilidade.empty:
        return

    mapa_cod_sub = {cod: sub for sub, dados in Codebook_Subtemas.items() for cod in dados.get("codigos_vinculados", [])}
    mapa_sub_tema = {sub: tema for tema, dados in Codebook_Temas.items() for sub in dados.get("subtemas_vinculados", [])}

    df_rastreabilidade['Subtema_Atribuido'] = df_rastreabilidade.get('Codigo_Atribuido', pd.Series(dtype=str)).map(mapa_cod_sub)
    df_rastreabilidade['Tema_Atribuido'] = df_rastreabilidade.get('Subtema_Atribuido', pd.Series(dtype=str)).map(mapa_sub_tema)
    
    linhas_codebook = []
    for tema, d_tema in Codebook_Temas.items():
        desc_tema = d_tema.get("descricao", "")
        for sub in d_tema.get("subtemas_vinculados", []):
            desc_sub = Codebook_Subtemas.get(sub, {}).get("descricao", "")
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evs = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                textos_amostra = "\n--- \n".join([f'"{e["citacao"]}" ({e["justificativa"]})' for e in evs[:3]])
                
                linhas_codebook.append({
                    "Tema": tema, 
                    "Descricao_Tema": desc_tema,
                    "Subtema": sub,
                    "Descricao_Subtema": desc_sub,
                    "Codigo": cod, 
                    "Descricao_Codigo": Codebook_Codigos.get(cod, {}).get("descricao", ""),
                    "Exemplos_Citacoes": textos_amostra
                })
    
    df_codebook = pd.DataFrame(linhas_codebook)
    
    df_evolucao = pd.DataFrame()
    if dados_evolucao:
        c_pre, s_pre, t_pre = dados_evolucao["codigos_pre"], dados_evolucao["subtemas_pre"], dados_evolucao["temas_pre"]
        c_pos, s_pos, t_pos = len(Codebook_Codigos), len(Codebook_Subtemas), len(Codebook_Temas)
        
        df_evolucao = pd.DataFrame([
            {"Métrica Estrutural": "Volume Total de Verbatims Extraídos (Fase 0)", "Fase Pré-Harmonização (Incremental)": dados_evolucao["verbatims"], "Fase Pós-Harmonização (Consolidado)": dados_evolucao["verbatims"], "Redução Líquida (%)": 0.0},
            {"Métrica Estrutural": "Densidade Total de Códigos Ativos (Fase 1)", "Fase Pré-Harmonização (Incremental)": c_pre, "Fase Pós-Harmonização (Consolidado)": c_pos, "Redução Líquida (%)": round(((c_pre - c_pos) / c_pre * 100), 2) if c_pre > 0 else 0.0},
            {"Métrica Estrutural": "Estruturas de Subtemas Mapeadas (Fase 2)", "Fase Pré-Harmonização (Incremental)": s_pre, "Fase Pós-Harmonização (Consolidado)": s_pos, "Redução Líquida (%)": round(((s_pre - s_pos) / s_pre * 100), 2) if s_pre > 0 else 0.0},
            {"Métrica Estrutural": "Eixos de Temas Macro Estruturantes (Fase 3)", "Fase Pré-Harmonização (Incremental)": t_pre, "Fase Pós-Harmonização (Consolidado)": t_pos, "Redução Líquida (%)": round(((t_pre - t_pos) / t_pre * 100), 2) if t_pre > 0 else 0.0}
        ])
    
    nome_arquivo = f"Resultado_Analise_Tematica_{sufixo_arquivo}.xlsx"
    caminho_saida = os.path.join(DIRETORIO_RELATORIO, nome_arquivo)
    
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df_rastreabilidade.to_excel(writer, sheet_name="Dados_Codificados", index=False)
        df_codebook.to_excel(writer, sheet_name="Codebook_Consolidado", index=False)
        
        # Injeção das Matrizes conforme status da iteração (Snapshot Duplo)
        if sufixo_arquivo == "Consolidado_Final":
            if Matriz_Saturacao_Antes is not None and not Matriz_Saturacao_Antes.empty:
                Matriz_Saturacao_Antes.to_excel(writer, sheet_name="Matriz_Saturacao_Antes", index=False)
            if Matriz_Saturacao_Apos is not None and not Matriz_Saturacao_Apos.empty:
                Matriz_Saturacao_Apos.to_excel(writer, sheet_name="Matriz_Saturacao_Apos", index=False)
        else:
            df_saturacao = gerar_matriz_saturacao_ampliada()
            if not df_saturacao.empty: 
                df_saturacao.to_excel(writer, sheet_name="Matriz_Saturacao", index=False)
                
        if not df_evolucao.empty:
            df_evolucao.to_excel(writer, sheet_name="Balanco_Fagocitose_Final", index=False)
            
def executar_fagocitose_final_consolidada():
    """
    [HIGH-LEVEL DOCUMENTATION]: MOTOR DE HARMONIZAÇÃO GLOBAL COM RESOLUÇÃO DE SUFIXOS
    Aplica expressões regulares para neutralizar marcações de colapso taxonômico (_, __)
    na camada exposta à LLM e realiza o mapeamento reverso determinístico sobre a base física.
    """
    global Codebook_Subtemas, Codebook_Temas, Codebook_Codigos
    print("\n" + "="*60 + "\n[FAGOCITOSE MACRO] INICIANDO HARMONIZAÇÃO GLOBAL DO CODEBOOK\n" + "="*60)
    
    # -------------------------------------------------------------------------
    # FASE A: Fagocitose de Subtemas (Proximidade Semântica Ampliada)
    # -------------------------------------------------------------------------
    subtemas_input = {}
    mapa_reverso_sub = {}
    
    for k, v in Codebook_Subtemas.items():
        if v.get("descricao"):
            k_limpo = re.sub(r'_+$', '', k).upper().strip()
            k_prompt = k_limpo
            idx = 1
            while k_prompt in subtemas_input:
                idx += 1
                k_prompt = f"{k_limpo} (VARIAÇÃO {idx})"
            subtemas_input[k_prompt] = v.get("descricao", "")
            mapa_reverso_sub[k_prompt] = k
            
    houve_fusao_sub = False
    
    if not subtemas_input:
        print("  [SKIP] Nenhum subtema disponível para análise na Fase A.")
    else:
        instrucao_sub = """Você é um Especialista Sénior em Engenharia de Processos e Auditoria de Sistemas de Trabalho. 
        Analise a lista de Subtemas e suas definições conceituais baseadas no funcionamento de normas corporativas.
        Sua tarefa é identificar redundâncias, sobreposições severas ou categorias que compartilhem proximidade semântica ampliada.
        Não se limite a sinônimos perfeitos. Se dois subtemas abordam dimensões complementares ou faces homólogas do mesmo gargalo operacional, agrupe-os.
        Retorne uma lista de mapeamentos indicando qual 'elemento_original' deve ser fundido em qual 'elemento_destino' sobrevivente. 
        Se a categoria for única e estruturalmente isolada, mapeie-a para ela mesma (REUSE)."""
        
        prompt_sub = f"SUBTEMAS ATUAIS DO ECCOSSISTEMA:\n{json.dumps(subtemas_input, ensure_ascii=False)}"
        fusoes_sub = invocar_llm(prompt_sub, list[FusaoItem], instrucao_sub, log_contexto="FAGOCITOSE GLOBAL | SUBTEMAS")
        
        for fusao in fusoes_sub:
            origem_prompt = fusao["elemento_original"].upper().strip()
            destino_prompt = fusao["elemento_destino"].upper().strip()
            
            origem = mapa_reverso_sub.get(origem_prompt)
            destino = mapa_reverso_sub.get(destino_prompt)
            
            if origem and destino and origem != destino and origem in Codebook_Subtemas and destino in Codebook_Subtemas:
                desc_origem = Codebook_Subtemas[origem].get("descricao", "")
                desc_destino = Codebook_Subtemas[destino].get("descricao", "")
                
                # Unificação de tokens otimizada na raiz descritiva
                nova_descricao = unificar_descricoes_llm(destino_prompt, [desc_destino, desc_origem], "SUBTEMA")
                
                registrar_e_executar_fagocitose("SUBTEMA", destino, [origem])
                Codebook_Subtemas[destino]["descricao"] = nova_descricao
                houve_fusao_sub = True
            
    # -------------------------------------------------------------------------
    # FASE B: Resolução de Ambiguidades de Vínculos Trans-Hierárquicos
    # -------------------------------------------------------------------------
    for tema, dados_tema in list(Codebook_Temas.items()):
        subs_atualizados = []
        for s in dados_tema.get("subtemas_vinculados", []):
            s_clean = s.upper().strip()
            if s_clean in Codebook_Subtemas and s_clean not in subs_atualizados:  
                subs_atualizados.append(s_clean)
        Codebook_Temas[tema]["subtemas_vinculados"] = subs_atualizados

    mapa_sub_temas = {}
    for tema, dados_tema in Codebook_Temas.items():
        for s in dados_tema.get("subtemas_vinculados", []):
            if s not in mapa_sub_temas: mapa_sub_temas[s] = []
            mapa_sub_temas[s].append(tema)
            
    for sub, lista_temas in mapa_sub_temas.items():
        if len(lista_temas) > 1:
            print(f"  [AMBIGUIDADE DETECTADA] Subtema '{sub}' está vinculado a múltiplos temas: {lista_temas}")
            instrucao_resolucao = "Você é um Analista Qualitativo Clássico especialista em engenharia de processos. Decida a qual único Tema este Subtema deve pertencer com base no cruzamento rigoroso de suas definições de escopo."
            temas_concorrentes_contexto = {t: Codebook_Temas[t].get("descricao", "Sem descrição.") for t in lista_temas if t in Codebook_Temas}
            
            prompt_res = (
                f"SUBTEMA: {sub}\n"
                f"DEFINIÇÃO: {Codebook_Subtemas[sub].get('descricao')}\n"
                "TEMAS CONCORRENTES E SEUS ESCOPOS CONCEITUAIS:\n"
                f"{json.dumps(temas_concorrentes_contexto, ensure_ascii=False, indent=4)}")
            resolvido = invocar_llm(prompt_res, MapeamentoResolvido, instrucao_resolucao, log_contexto="RESOLUÇÃO AMBIGUIDADE")
            
            tema_escolhido = resolvido.get("tema_final_alocado", "").upper().strip()
            for t in lista_temas:
                if t != tema_escolhido and sub in Codebook_Temas[t]["subtemas_vinculados"]:  
                    Codebook_Temas[t]["subtemas_vinculados"].remove(sub)
            if sub not in Codebook_Temas[tema_escolhido]["subtemas_vinculados"]:
                Codebook_Temas[tema_escolhido]["subtemas_vinculados"].append(sub)

    # -------------------------------------------------------------------------
    # FASE C: Fagocitose de Temas Macroestruturantes (Proximidade Semântica Ampliada)
    # -------------------------------------------------------------------------
    temas_input = {}
    mapa_reverso_tema = {}
    
    for k, v in Codebook_Temas.items():
        if v.get("descricao"):
            k_limpo = re.sub(r'_+$', '', k).upper().strip()
            k_prompt = k_limpo
            idx = 1
            while k_prompt in temas_input:
                idx += 1
                k_prompt = f"{k_limpo} (VARIAÇÃO {idx})"
            temas_input[k_prompt] = v.get("descricao", "")
            mapa_reverso_tema[k_prompt] = k
            
    houve_fusao_tema = False
    
    if not temas_input:
        print("  [SKIP] Nenhum tema disponível para análise na Fase C.")
    else:
        instrucao_tema = """Você é um Especialista Sênior em Auditoria de Processos. Analise a camada macro de Temas Estruturantes. 
        Se dois macro-constructos convergirem para a mesma narrativa teórica global por proximidade semântica ampliada, ordene a fusão. Retorne os acoplamentos necessários."""
        
        prompt_tema = f"TEMAS ATUAIS DO ECOSSISTEMA:\n{json.dumps(temas_input, ensure_ascii=False)}"
        fusoes_tema = invocar_llm(prompt_tema, list[FusaoItem], instrucao_tema, log_contexto="FAGOCITOSE GLOBAL | TEMAS")
        
        for fusao in fusoes_tema:
            origem_prompt = fusao["elemento_original"].upper().strip()
            destino_prompt = fusao["elemento_destino"].upper().strip()
            
            origem = mapa_reverso_tema.get(origem_prompt)
            destino = mapa_reverso_tema.get(destino_prompt)
            
            if origem and destino and origem != destino and origem in Codebook_Temas and destino in Codebook_Temas:
                desc_origem = Codebook_Temas[origem].get("descricao", "")
                desc_destino = Codebook_Temas[destino].get("descricao", "")
                
                nova_descricao = unificar_descricoes_llm(destino_prompt, [desc_destino, desc_origem], "TEMA")
                
                registrar_e_executar_fagocitose("TEMA", destino, [origem])
                Codebook_Temas[destino]["descricao"] = nova_descricao
                houve_fusao_tema = True

In [14]:
# 6. ENGENHARIA DE RELATÓRIOS, MATRIZES E VISUALIZAÇÃO PANORÂMICA
# ==============================================================================
# DESCRIÇÃO ARQUITETURAL:
# Responsável pela materialização dos dados em disco para consumo humano.
# 1. Exportação Relacional (Excel): Constrói a rastreabilidade Bottom-Up 
#    (Código -> Subtema -> Tema), consolida o Codebook com as Descrições Mestres 
#    e gera a Matriz de Saturação Teórica.
# 2. Visualização Científica (Grafos): Implementa um algoritmo de renderização 
#    de Grafo Direcionado (DiGraph) para gerar um Dendrograma Hierárquico. 
#    Utiliza cálculo geométrico para alinhamento Top-Down (Esquerda para Direita) 
#    e escala de canal Alpha (transparência) para representar profundidade analítica.
# ==============================================================================

def exportar_artefatos(sufixo_arquivo="", dados_evolucao: dict = None):
    """
    Gera a documentação relacional do estudo em formato de planilha Excel.
    Se dados_evolucao for fornecido, acopla a aba executiva de balanço Pré vs Pós-Harmonização.
    """
    df_rastreabilidade = pd.DataFrame(Rastreabilidade_Base)
    if df_rastreabilidade.empty:
        return

    mapa_cod_sub = {cod: sub for sub, dados in Codebook_Subtemas.items() for cod in dados.get("codigos_vinculados", [])}
    mapa_sub_tema = {sub: tema for tema, dados in Codebook_Temas.items() for sub in dados.get("subtemas_vinculados", [])}

    df_rastreabilidade['Subtema_Atribuido'] = df_rastreabilidade.get('Codigo_Atribuido', pd.Series(dtype=str)).map(mapa_cod_sub)
    df_rastreabilidade['Tema_Atribuido'] = df_rastreabilidade.get('Subtema_Atribuido', pd.Series(dtype=str)).map(mapa_sub_tema)
    
    linhas_codebook = []
    for tema, d_tema in Codebook_Temas.items():
        desc_tema = d_tema.get("descricao", "")
        for sub in d_tema.get("subtemas_vinculados", []):
            desc_sub = Codebook_Subtemas.get(sub, {}).get("descricao", "")
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evs = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                textos_amostra = "\n--- \n".join([f'"{e["citacao"]}" ({e["justificativa"]})' for e in evs[:3]])
                
                linhas_codebook.append({
                    "Tema": tema, 
                    "Descricao_Tema": desc_tema,
                    "Subtema": sub,
                    "Descricao_Subtema": desc_sub,
                    "Codigo": cod, 
                    "Descricao_Codigo": Codebook_Codigos.get(cod, {}).get("descricao", ""),
                    "Exemplos_Citacoes": textos_amostra
                })
    
    df_codebook = pd.DataFrame(linhas_codebook)
    if sufixo_arquivo == "Consolidado_Final" and Matriz_Saturacao_Real is not None:
        df_saturacao = Matriz_Saturacao_Real
    else:
        df_saturacao = gerar_matriz_saturacao_ampliada()
    
    # [HIGH-LEVEL DOCUMENTATION]: CONSTRUÇÃO DA ABA DE EVOLUÇÃO TAXONÔMICA GLOBAL
    df_evolucao = pd.DataFrame()
    if dados_evolucao:
        c_pre, s_pre, t_pre = dados_evolucao["codigos_pre"], dados_evolucao["subtemas_pre"], dados_evolucao["temas_pre"]
        c_pos, s_pos, t_pos = len(Codebook_Codigos), len(Codebook_Subtemas), len(Codebook_Temas)
        
        df_evolucao = pd.DataFrame([
            {"Métrica Estrutural": "Volume Total de Verbatims Extraídos (Fase 0)", "Fase Pré-Harmonização (Incremental)": dados_evolucao["verbatims"], "Fase Pós-Harmonização (Consolidado)": dados_evolucao["verbatims"], "Redução Líquida (%)": 0.0},
            {"Métrica Estrutural": "Densidade Total de Códigos Ativos (Fase 1)", "Fase Pré-Harmonização (Incremental)": c_pre, "Fase Pós-Harmonização (Consolidado)": c_pos, "Redução Líquida (%)": round(((c_pre - c_pos) / c_pre * 100), 2) if c_pre > 0 else 0.0},
            {"Métrica Estrutural": "Estruturas de Subtemas Mapeadas (Fase 2)", "Fase Pré-Harmonização (Incremental)": s_pre, "Fase Pós-Harmonização (Consolidado)": s_pos, "Redução Líquida (%)": round(((s_pre - s_pos) / s_pre * 100), 2) if s_pre > 0 else 0.0},
            {"Métrica Estrutural": "Eixos de Temas Macro Estruturantes (Fase 3)", "Fase Pré-Harmonização (Incremental)": t_pre, "Fase Pós-Harmonização (Consolidado)": t_pos, "Redução Líquida (%)": round(((t_pre - t_pos) / t_pre * 100), 2) if t_pre > 0 else 0.0}
        ])
    
    nome_arquivo = f"Resultado_Analise_Tematica_{sufixo_arquivo}.xlsx"
    caminho_saida = os.path.join(DIRETORIO_RELATORIO, nome_arquivo)
    
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df_rastreabilidade.to_excel(writer, sheet_name="Dados_Codificados", index=False)
        df_codebook.to_excel(writer, sheet_name="Codebook_Consolidado", index=False)
        if not df_saturacao.empty: 
            df_saturacao.to_excel(writer, sheet_name="Matriz_Saturacao", index=False)
        if not df_evolucao.empty:
            df_evolucao.to_excel(writer, sheet_name="Balanco_Fagocitose_Final", index=False)

def gerar_matriz_saturacao_ampliada(estrito_imagem: bool = True) -> pd.DataFrame:
    """
    [HIGH-LEVEL DOCUMENTATION]: CALCULADORA CRONOLÓGICA DE SATURAÇÃO EMPÍRICA
    Mapeia retrospectivamente o surgimento de novos conceitos por arquivo.
    Se estrito_imagem=True, formata a saída com a estrutura exata de 4 colunas
    solicitada, incluindo a linha de fechamento de quórum (Total).
    """
    if not Rastreabilidade_Base: 
        return pd.DataFrame()
        
    mapa_cod_sub = {cod: sub for sub, dados in Codebook_Subtemas.items() for cod in dados.get("codigos_vinculados", [])}
    mapa_sub_tema = {sub: tema for tema, dados in Codebook_Temas.items() for sub in dados.get("subtemas_vinculados", [])}

    arquivos_ordem = []
    for item in Rastreabilidade_Base:
        if item["Arquivo_Origem"] not in arquivos_ordem: 
            arquivos_ordem.append(item["Arquivo_Origem"])

    codigos_vistos = set()
    subtemas_vistos = set()
    temas_vistos = set()
    matriz_linhas = []

    for arquivo in arquivos_ordem:
        cods_neste_arquivo = {item["Codigo_Atribuido"] for item in Rastreabilidade_Base if item["Arquivo_Origem"] == arquivo and item.get("Codigo_Atribuido")}
        novos_cods = novos_subs = novos_temas = 0

        for cod in cods_neste_arquivo:
            if cod not in codigos_vistos: 
                novos_cods += 1
                codigos_vistos.add(cod)
            
            sub = mapa_cod_sub.get(cod)
            if sub and sub not in subtemas_vistos: 
                novos_subs += 1
                subtemas_vistos.add(sub)
            
            tema = mapa_sub_tema.get(sub) if sub else None
            if tema and tema not in temas_vistos: 
                novos_temas += 1
                temas_vistos.add(tema)

        if estrito_imagem:
            matriz_linhas.append({
                "Nome_Arquivo": arquivo,
                "Novos_Codigos": novos_cods,
                "Novos_Subtemas": novos_subs,
                "Novos_Temas": novos_temas
            })
        else:
            matriz_linhas.append({
                "Nome_Arquivo": arquivo,
                "Novos_Codigos": novos_cods, "Total_Codigos": len(codigos_vistos),
                "Novos_Subtemas": novos_subs, "Total_Subtemas": len(subtemas_vistos),
                "Novos_Temas": novos_temas, "Total_Temas": len(temas_vistos)
            })
        
    df = pd.DataFrame(matriz_linhas)
    
    if estrito_imagem and not df.empty:
        # Injeção dinâmica da linha de fechamento analítico (Total)
        linha_total = pd.DataFrame([{
            "Nome_Arquivo": "Total",
            "Novos_Codigos": df["Novos_Codigos"].sum(),
            "Novos_Subtemas": df["Novos_Subtemas"].sum(),
            "Novos_Temas": df["Novos_Temas"].sum()
        }])
        df = pd.concat([df, linha_total], ignore_index=True)
        
    return df

def gerar_graficos_cientificos(sufixo_iteracao="Final", filtrar_relevantes=True):
    """
    Gera a representação visual macro do modelo teórico resultante.
    
    Lógica de Negócio e Algoritmo Gráfico:
      1. Instancia um Grafo Direcionado (nx.DiGraph).
      2. Mapeia posições (X, Y) programaticamente. 
         - X=0 (Temas), X=1 (Subtemas), X=2 (Códigos).
         - Y é decrementado linearmente pelas 'folhas' (Códigos) para evitar sobreposição.
      3. Nós superiores (Subtemas e Temas) têm seu eixo Y calculado como o 
         ponto médio (média aritmética) do eixo Y de seus respectivos nós filhos.
      4. Aplica codificação visual baseada em heurísticas de percepção:
         - Matiz (Color): Diferencia os Temas macroestruturantes (Palette tab20).
         - Saturação (Alpha): Diferencia o nível de abstração (Tema=95%, Subtema=55%, Código=25%).
        Parâmetros:
        - sufixo_iteracao (str): Identificador numérico anexado ao nome do arquivo de saída.
            Garante a preservação histórica do estado da árvore categórica para trilha de auditoria.
    """
    print(f"\n--- INICIANDO GERAÇÃO DE MAPA TEMÁTICO (FILTRADO={filtrar_relevantes}) ---")
        
    G = nx.DiGraph()
    pos = {}
    node_colors = {}
    node_alphas = {}
    node_levels = {} # [NOVO] Dicionário para rastrear a hierarquia do nó
    
    # Definição de paleta para diferenciação categórica horizontal
    cmap = plt.get_cmap("tab20")
    temas = list(Codebook_Temas.keys())
    
    y_atual = 0
    largura_quebra = 40 # Limite de caracteres para quebra de linha nas caixas do grafo
    
    # =====================================================================
    # MOTOR DE CÁLCULO DE COORDENADAS ESPACIAIS (GRID DISTRIBUÍDO)
    # =====================================================================
    largura_quebra_tema = 30
    largura_quebra_codigo = 30 # Mais estreito para caber lado a lado
    max_colunas_codigos = 4    # Quantidade de códigos na horizontal
    passo_x_codigo = 0.8       # Distância horizontal entre códigos
    passo_y_codigo = 0.8       # Distância vertical entre linhas do grid

    for i, tema in enumerate(temas):
        cor_base = cmap(i % 20)
        tema_lbl = "\n".join(textwrap.wrap(tema, largura_quebra_tema))
        
        if tema_lbl not in pos:
            G.add_node(tema_lbl)
            node_colors[tema_lbl] = cor_base
            node_alphas[tema_lbl] = 0.95
            node_levels[tema_lbl] = "tema"
        
        subtemas = Codebook_Temas[tema].get("subtemas_vinculados", [])
        y_tema_start = y_atual
        
        for sub in subtemas:
            sub_lbl = "\n".join(textwrap.wrap(sub, largura_quebra_tema))
            
            if sub_lbl not in pos:
                G.add_node(sub_lbl)
                node_colors[sub_lbl] = cor_base
                node_alphas[sub_lbl] = 0.55
                node_levels[sub_lbl] = "subtema"
            
            G.add_edge(tema_lbl, sub_lbl)
            
            codigos = Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", [])
            has_new_codes = False
            y_min_neste_subtema = y_atual
            
            # FILTRAGEM ALGORÍTMICA CONDICIONAL: Duplo canal de visualização
            if filtrar_relevantes:
                codigos_filtrados = [c for c in codigos if int(Codebook_Codigos.get(c, {}).get("nivel_relevancia_meta", 0)) >= 3]
            else:
                codigos_filtrados = codigos
            
            # Distribuição em Matriz Bidimensional (Grid) utilizando a lista resultante
            for idx, cod in enumerate(codigos_filtrados):
                cod_lbl = "\n".join(textwrap.wrap(cod, largura_quebra_codigo))
                
                if cod_lbl not in pos:
                    G.add_node(cod_lbl)
                    node_colors[cod_lbl] = cor_base
                    node_alphas[cod_lbl] = 0.55
                    node_levels[cod_lbl] = "codigo"                    
                    coluna = idx % max_colunas_codigos
                    linha = idx // max_colunas_codigos
                    
                    pos_x = 2.5 + (coluna * passo_x_codigo)
                    pos_y = y_atual - (linha * passo_y_codigo)
                    
                    pos[cod_lbl] = (pos_x, pos_y)
                    
                    if pos_y < y_min_neste_subtema:
                        y_min_neste_subtema = pos_y
                        
                    has_new_codes = True
                    
                G.add_edge(sub_lbl, cod_lbl)
            
            # Centralização do Subtema e avanço do ponteiro Y
            if has_new_codes:
                pos[sub_lbl] = (1.0, (y_atual + y_min_neste_subtema) / 2.0)
                y_atual = y_min_neste_subtema - 1.2 # Margem de respiro para o próximo subtema
            elif sub_lbl not in pos: 
                pos[sub_lbl] = (1.0, y_atual)
                y_atual -= 1.2
        
        # Centralização Geométrica do Tema
        if y_tema_start != y_atual:
            # Compensa a última margem de respiro inserida no loop de subtemas
            pos[tema_lbl] = (0, (y_tema_start + y_atual + 1.2) / 2.0) 
        elif tema_lbl not in pos:
            pos[tema_lbl] = (0, y_atual)
            y_atual -= 1.5

    if not G.nodes():
        print("  [AVISO] Dados insuficientes para gerar o mapa panorâmico.")
        return

    # =====================================================================
    # MOTOR DE RENDERIZAÇÃO MATPLOTLIB
    # =====================================================================
    altura_canvas = max(12.0, abs(y_atual) * 0.9)
    largura_canvas = 40.0 
    plt.figure(figsize=(largura_canvas, altura_canvas))
    
    # zorder=1: Força as linhas para a camada de fundo
    nx.draw_networkx_edges(G, pos, edge_color="#CCCCCC", arrows=True, arrowstyle="-|>", arrowsize=10, width=1.2)
    
    for node in G.nodes():
        x, y = pos[node]
        cor = node_colors.get(node, "#DDDDDD")
        alpha = node_alphas.get(node, 0.5)
        nivel = node_levels.get(node, "codigo") # Recupera a classificação estrutural
        
        rgba_color = mcolors.to_rgba(cor, alpha=alpha)
        
        # Lógica de Controle Tipográfico por Nível
        if nivel == "tema":
            tamanho_fonte = 16
            peso_fonte = 'black' # Extra-bold para máximo destaque
        elif nivel == "subtema":
            tamanho_fonte = 13
            peso_fonte = 'bold'
        else:
            tamanho_fonte = 10
            peso_fonte = 'bold'
        
        plt.text(x, y, node, 
                 fontsize=tamanho_fonte, 
                 fontweight=peso_fonte,
                 fontfamily='sans-serif',
                 ha='center', va='center',
                 zorder=10, 
                 bbox=dict(boxstyle="round,pad=0.6", facecolor=rgba_color, edgecolor="#555555", linewidth=1.2))

    plt.xlim(-0.5, 3.0 + (max_colunas_codigos * passo_x_codigo))
    plt.ylim(y_atual - 1, 1.5) 
    plt.axis("off")
    
    # Montagem da legenda estrutural de profundidade
    legendas = [
        mpatches.Patch(facecolor='gray', alpha=0.95, edgecolor='black', label='Tema (Nível 1)'),
        mpatches.Patch(facecolor='gray', alpha=0.55, edgecolor='black', label='Subtema (Nível 2)'),
        mpatches.Patch(facecolor='gray', alpha=0.25, edgecolor='black', label='Código (Nível 3)')
    ]
    plt.legend(handles=legendas, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02), 
               title="Árvore Hierárquica Relacional (Cada matiz de cor representa 1 Tema Macroestruturante)", 
               title_fontsize=11)
    
    if filtrar_relevantes:
        nome_arquivo = f"Mapa_Tematico_Panoramico_{sufixo_iteracao}.png"
    else:
        nome_arquivo = f"Mapa_Tematico_Panoramico_{sufixo_iteracao}_cauda_longa.png"
        
    caminho_saida = os.path.join(DIRETORIO_GRAFICOS, nome_arquivo)
    plt.savefig(caminho_saida, format="png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Mapa Panorâmico exportado em: {caminho_saida}")

def gerar_grafico_frequencia_caixas():
    """
    Gera representação Treemap.
    Objetivo: Mapear a densidade empírica onde a área geométrica da caixa 
    é estritamente proporcional ao volume de incidência (repetições) do código.
    """
    print("\n--- INICIANDO GERAÇÃO DE MAPA DE CAIXAS (TREEMAP) ---")
    
    labels = []
    sizes = []
    colors = []
    cmap = plt.get_cmap("tab20")
    
    for i, (tema, d_tema) in enumerate(Codebook_Temas.items()):
        cor_base = cmap(i % 20)
        for sub in d_tema.get("subtemas_vinculados", []):
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evidencias = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                frequencia = len(evidencias)
                if frequencia > 0:
                    rotulo = f"{cod}\n(n={frequencia})"
                    labels.append(rotulo)
                    sizes.append(frequencia)
                    colors.append(mcolors.to_rgba(cor_base, alpha=0.6))
    
    if not sizes:
        print("  [AVISO] Dados insuficientes para o Treemap.")
        return

    plt.figure(figsize=(16, 10))
    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8,
                  text_kwargs={'fontsize': 10, 'fontweight': 'bold', 'wrap': True})
    
    plt.axis('off')
    plt.title("Densidade Temática: Proporção de Códigos por Saturação de Evidências", fontsize=14, fontweight='bold', pad=20)
    
    caminho_saida = os.path.join(DIRETORIO_GRAFICOS, "Mapa_Frequencia_Caixas.png")
    plt.savefig(caminho_saida, format="png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Treemap exportado em: {caminho_saida}")

def gerar_grafico_hierarquico_proporcional():
    """
    Gera representação hierárquica proporcional usando go.Icicle.
    A paleta de cores foi antecipada para permitir a estilização inline dos blocos de códigos.
    """
    print("\n--- INICIANDO GERAÇÃO DE MAPA HIERÁRQUICO (MODO TABELA PROPORCIONAL) ---")
    
    freq_por_tema = {}
    freq_por_subtema = {}
    codigos_agrupados = {}
    
    # 1. Definição antecipada do mapeamento de cores para alimentar as strings HTML
    cmap = px.colors.qualitative.Pastel
    temas_unicos = list(Codebook_Temas.keys())
    color_map = {tema: cmap[i % len(cmap)] for i, tema in enumerate(temas_unicos)}
    
    for tema, d_tema in Codebook_Temas.items():
        cor_tema = color_map.get(tema, "#CCCCCC")
        soma_tema = 0
        for sub in d_tema.get("subtemas_vinculados", []):
            soma_sub = 0
            lista_codigos = []
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evidencias = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                freq = len(evidencias)
                if freq > 0:
                    soma_sub += freq
                    # Injeção de estilo inline para emular caixas coloridas nos códigos individuais
                    lista_codigos.append(
                        f"<span style='background-color: {cor_tema}; padding: 4px 8px; "
                        f"border: 1px solid #555555; border-radius: 4px; display: inline-block; "
                        f"margin: 2px;'>{cod} (n={freq})</span>"
                    )
            
            if soma_sub > 0:
                freq_por_subtema[f"T_{tema}_S_{sub}"] = soma_sub
                
                chunk_size = 2 
                linhas = []
                for i in range(0, len(lista_codigos), chunk_size):
                    linhas.append(" &nbsp;&nbsp; ".join(lista_codigos[i:i+chunk_size]))
                
                codigos_agrupados[f"T_{tema}_S_{sub}_C"] = "<br><br>".join(linhas)
                soma_tema += soma_sub
                
        if soma_tema > 0:
            freq_por_tema[f"T_{tema}"] = soma_tema

    if not freq_por_tema:
        print("  [AVISO] Dados insuficientes para o gráfico hierárquico.")
        return

    ids = []
    labels = []
    parents = []
    values = []

# Construção: Nível 1 - Temas
    for tema_id, freq in freq_por_tema.items():
        ids.append(tema_id)
        # BURLA DE LIMITE HORIZONTAL: Empilhamento vertical de palavras
        # nome_tema = tema_id.replace("T_", "").upper()
        nome_tema = tema_id[2:].upper()
        labels.append(f"<b>{nome_tema}</b><br><br>(n={freq})")
        parents.append("") 
        values.append(freq)

    # Construção: Nível 2 - Subtemas
    for sub_id, freq in freq_por_subtema.items():
        ids.append(sub_id)
        # BURLA DE LIMITE HORIZONTAL: Empilhamento vertical de palavras
        nome_sub = sub_id.split("_S_")[1].title().replace(" ", "<br>") 
        labels.append(f"<b>{nome_sub}</b><br><br>(n={freq})")
        tema_parent = sub_id.split("_S_")[0]
        parents.append(tema_parent)
        values.append(freq)

    # Construção: Nível 3 - Códigos (Formatação HTML preservada)
    for cod_id, texto_html in codigos_agrupados.items():
        ids.append(cod_id)
        labels.append(texto_html)
        # sub_parent = cod_id.replace("_C", "")
        sub_parent = cod_id[:-2]

        parents.append(sub_parent)
        values.append(freq_por_subtema[sub_parent])

    # Força o contentor de fundo da última coluna a ser transparente
    colors = []
    for node_id in ids:
        if node_id.endswith("_C"):
            colors.append("rgba(0,0,0,0)") 
        else:
            tema_base = node_id.split("_S_")[0]
            # colors.append(color_map.get(tema_base.replace("T_", ""), "#CCCCCC"))
            colors.append(color_map.get(tema_base[2:], "#CCCCCC"))

    fig = go.Figure(go.Icicle(
        ids=ids,
        labels=labels,
        parents=parents,
        values=values,
        branchvalues="total",
        marker=dict(colors=colors, line=dict(color='#FFFFFF', width=1.5)),
        textinfo="label",
        # Reduzido de 30 para 14. Um valor menor impede que o algoritmo do Plotly 
        # entre em "pânico" tentando espremer textos gigantes, resultando em uma 
        # renderização nativa mais nítida e equilibrada.
        textfont=dict(size=14, color="#111111"), 
        pathbar=dict(visible=False),
        tiling=dict(pad=2) 
    ))

    annotations = [
        dict(x=0.16, y=1.03, xref='paper', yref='paper', text="<b>TEMA</b>", showarrow=False, font=dict(size=14), xanchor="center"),
        dict(x=0.50, y=1.03, xref='paper', yref='paper', text="<b>SUBTEMA</b>", showarrow=False, font=dict(size=14), xanchor="center"),
        dict(x=0.83, y=1.03, xref='paper', yref='paper', text="<b>CÓDIGOS E EVIDÊNCIAS</b>", showarrow=False, font=dict(size=14), xanchor="center")
    ]

    fig.update_layout(
        margin=dict(t=50, l=10, r=10, b=10),
        annotations=annotations,
        # REDUZIDO: A largura total foi diminuída (de 1400 para 1200) para forçar as colunas a ficarem mais próximas. 
        # Se os textos dos códigos começarem a quebrar de forma indesejada, aumente para 1250 ou 1300.
        width=1200, 
        height=900
    )

    caminho_saida_png = os.path.join(DIRETORIO_GRAFICOS, "Mapa_Hierarquico_Proporcional.png")
    fig.write_image(caminho_saida_png, scale=2)
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Gráfico de áreas exportado em: {caminho_saida_png}")

In [15]:
# 7. Orquestrador: Novo fluxo de controle incremental. Lê as fontes da verdade, processa as 6
# etapas apenas para o arquivo da iteração e grava as modificações. 
# ==============================================================================

def orquestrar_pipeline_incremental():
    """
    [HIGH-LEVEL DOCUMENTATION]: ORQUESTRAÇÃO DE FLUXO METODOLÓGICO SEPARADO
    Isola o laço incremental estritamente para a extração (Fase 0) e codificação (Fase 1).
    Executa de forma holística pós-corpus as fases de abstração alta e congelamento duplo.
    """
    terminal_original = sys.stdout
    
    class LoggerDuplo(object):
        def __init__(self, caminho_arquivo, terminal_alvo):
            self.terminal = terminal_alvo
            self.log = open(caminho_arquivo, "a", encoding="utf-8")
            
        def write(self, mensagem):
            self.terminal.write(mensagem)
            self.log.write(mensagem)
            self.log.flush()
            
        def flush(self):
            self.terminal.flush()
            self.log.flush()

    sys.stdout = LoggerDuplo(ARQUIVO_LOG_CONSOLE, terminal_original)
    logger_inst = sys.stdout
    
    try:
        print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] --- INICIALIZAÇÃO DO FLUXO DE AUDITORIA DE CONSOLE ---")

        arquivos_alvo = ordenar_arquivos_cronologicamente(DIRETORIO_ENTREVISTAS)
        if not arquivos_alvo:
            raise FileNotFoundError(f"Nenhum arquivo válido (.txt) localizado em {DIRETORIO_ENTREVISTAS}.")

        # LAÇO INCREMENTAL: Focado estritamente no pilar empírico (Fase 0 e Fase 1)
        for idx, arquivo in enumerate(arquivos_alvo, start=1):
            carregar_estado_json()
            
            # Checagem de processamento prévio do arquivo na base de rastreabilidade
            arquivo_ja_codificado = any(ev.get("Arquivo_Origem") == arquivo for ev in Rastreabilidade_Base)
            if arquivo_ja_codificado:
                print(f"[{arquivo}] Iteração {idx} (Fases 0 e 1) já consolidada em checkpoint. Avançando...")
                continue
                
            print(f"\n{'='*60}\nINICIANDO PROCESSAMENTO CRONOLÓGICO: {arquivo} (Iteração {idx})\n{'='*60}")
            
            print("\n[ETAPA 1] Codificação Primária e Extração Inédita por Bloco...")
            executar_fase_1(arquivo)
            salvar_estado_json() 
            
            print("\n[ETAPA 2] Síntese de Novos Códigos Atômicos...")
            executar_fase_intermediaria_1()
            salvar_estado_json()

        print("\n" + "="*60 + "\n--- LAÇO INCREMENTAL DE ENTREVISTAS CONCLUÍDO ---\n" + "="*60)
        
        # ABSTRAÇÃO HOLÍSTICA GLOBAL (Executada uma única vez após fechamento completo do corpus)
        carregar_estado_json()
        print("\n[ETAPA 3] Agrupamento Global: Códigos Órfãos -> Subtemas...")
        executar_fase_2()
        salvar_estado_json()
        
        print("\n[ETAPA 4] Síntese Global de Escopo de Subtemas...")
        executar_fase_intermediaria_2()
        salvar_estado_json()
        
        print("\n[ETAPA 5] Agrupamento Global: Subtemas Órfãos -> Temas...")
        executar_fase_3()
        salvar_estado_json()
        
        print("\n[ETAPA 6] Síntese Global de Escopo de Temas Macroestruturantes...")
        executar_fase_intermediaria_3()
        salvar_estado_json()

        global Matriz_Saturacao_Antes, Matriz_Saturacao_Apos
        
        # SNAPSHOT 1: Congelamento estrutural da curva empírica incremental antes da fagocitose
        Matriz_Saturacao_Antes = gerar_matriz_saturacao_ampliada(estrito_imagem=True)
        
        dados_pre_fagocitose = {
            "verbatims": len(Rastreabilidade_Base),
            "codigos_pre": len(Codebook_Codigos),
            "subtemas_pre": len(Codebook_Subtemas),
            "temas_pre": len(Codebook_Temas)
        }
        
        # Saneamento e Harmonização de proximidade semântica pós-corpus
        executar_fagocitose_final_consolidada()
        sanear_hierarquia_quoruns()
        salvar_estado_json()
        
        # SNAPSHOT 2: Recálculo retrospectivo limpo sobre a árvore final higienizada
        Matriz_Saturacao_Apos = gerar_matriz_saturacao_ampliada(estrito_imagem=True)
        
        print("\n[ETAPA FINAL] Exportando Relatórios Consolidados e Artefatos Visuais...")
        exportar_artefatos("Consolidado_Final", dados_evolucao=dados_pre_fagocitose)
        gerar_graficos_cientificos(sufixo_iteracao="Consolidado_Final", filtrar_relevantes=True)
        gerar_graficos_cientificos(sufixo_iteracao="Consolidado_Final", filtrar_relevantes=False)
        gerar_grafico_frequencia_caixas()
        gerar_grafico_hierarquico_proporcional()
        
        print("\n" + "="*60 + "\n--- PROCESSAMENTO INTEGRAL E HARMONIZAÇÃO CONCLUÍDOS ---\n" + "="*60)
        
    finally:
        try:
            logger_inst.log.close()
        except Exception:
            pass
        sys.stdout = terminal_original

In [ ]:
# 8. EXECUTAR ANÁLISE COM LLM

if __name__ == "__main__":
    try:
        orquestrar_pipeline_incremental()
        
        # SINAL DE CONCLUSÃO (SUCESSO): Tom único, longo e agudo
        import winsound
        winsound.Beep(2000, 1500)
        print("\n[ALERTA SONORO] Execução concluída com sucesso.")
        
    except Exception as e:
        # SINAL DE PARADA (ERRO/FALHA): Sequência de 3 bipes rápidos e graves
        import winsound
        for _ in range(3):
            winsound.Beep(1000, 250)
            time.sleep(0.05)
        print("\n[ALERTA SONORO] Execução interrompida por falha crítica.")
        
        # PRESERVAÇÃO DA DIRETRIZ: O erro NÃO é mascarado.
        # Reerguemos a exceção para que o Python quebre explicitamente no console.
        raise e


[2026-06-22 18:14:19] --- INICIALIZAÇÃO DO FLUXO DE AUDITORIA DE CONSOLE ---
[Entrevistado A1.txt] Iteração 1 (Fases 0 e 1) já consolidada em checkpoint. Avançando...

INICIANDO PROCESSAMENTO CRONOLÓGICO: Entrevistado A2.txt (Iteração 2)

[ETAPA 1] Codificação Primária e Extração Inédita por Bloco...
    [FASE 0] Extraindo verbatims estruturados inéditos no Bloco 1/8...


In [ ]:
# 9. 2ª RODADA SEQUÊNCIAL

DIRETORIO_CHECKPOINTS = os.path.join(DIRETORIO_ENTREVISTAS, "Checkpoints_2")
DIRETORIO_CACHE_F1 = os.path.join(DIRETORIO_CHECKPOINTS, "Fase1_Chunks_2")
DIRETORIO_GRAFICOS = os.path.join(DIRETORIO_ENTREVISTAS, "Graficos_2")
DIRETORIO_RELATORIO = os.path.join(DIRETORIO_ENTREVISTAS, "Relatorio_Final_2")

ARQUIVO_LOG_PROMPTS = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_prompts_2.md")

ARQUIVO_CODEBOOK_CODIGOS = os.path.join(DIRETORIO_CHECKPOINTS, "1_codebook_codigos_2.json")
ARQUIVO_CODEBOOK_SUBTEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "2_codebook_subtemas_2.json")
ARQUIVO_CODEBOOK_TEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "3_codebook_temas_2.json")
ARQUIVO_RASTREABILIDADE = os.path.join(DIRETORIO_CHECKPOINTS, "0_rastreabilidade_base_2.json")
ARQUIVO_HISTORICO_FUSOES = os.path.join(DIRETORIO_CHECKPOINTS, "historico_fagocitose_2.json")
ARQUIVO_LOG_CONSOLE = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_console_2.log")

# ASSEGURAÇÃO DE DIRETÓRIOS FÍSICOS DA SEGUNDA RODADA
os.makedirs(DIRETORIO_CHECKPOINTS, exist_ok=True)
os.makedirs(DIRETORIO_CACHE_F1, exist_ok=True)
os.makedirs(DIRETORIO_GRAFICOS, exist_ok=True)
os.makedirs(DIRETORIO_RELATORIO, exist_ok=True)

# Chamada da 2ª Rodada
if __name__ == "__main__":
    try:
        orquestrar_pipeline_incremental()
        
        # SINAL DE CONCLUSÃO (SUCESSO): Tom único, longo e agudo
        import winsound
        winsound.Beep(2000, 1500)
        print("\n[ALERTA SONORO] Execução concluída com sucesso.")
        
    except Exception as e:
        # SINAL DE PARADA (ERRO/FALHA): Sequência de 3 bipes rápidos e graves
        import winsound
        for _ in range(3):
            winsound.Beep(1000, 250)
            time.sleep(0.05)
        print("\n[ALERTA SONORO] Execução interrompida por falha crítica.")
        
        # PRESERVAÇÃO DA DIRETRIZ: O erro NÃO é mascarado.
        # Reerguemos a exceção para que o Python quebre explicitamente no console.
        raise e

In [ ]:
# # 10. Reparação de Dados (Rodar para recuperar padrões anteriores)

# def reparar_dados_corrompidos_offline():
#     """
#     [HIGH-LEVEL DOCUMENTATION]: REPARADOR ALGORÍTMICO DE ESQUEMA TAXONÔMICO
#     Saneia retroativamente as contaminações estruturais geradas por regras lógicas
#     antigas nos arquivos JSON locais, sem realizar chamadas de LLM.
#     """
#     if not os.path.exists(ARQUIVO_CODEBOOK_TEMAS) or not os.path.exists(ARQUIVO_CODEBOOK_SUBTEMAS):
#         print("[ERRO] Arquivos JSON mestre não localizados para reparação.")
#         return

#     with open(ARQUIVO_CODEBOOK_TEMAS, 'r', encoding='utf-8') as f:
#         temas = json.load(f)
#     with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'r', encoding='utf-8') as f:
#         subtemas = json.load(f)

#     print("[REPARAÇÃO] Iniciando saneamento algorítmico dos arquivos atuais...")

#     # 1. Remove códigos folhas que foram erradamente injetados no nível de Subtema
#     for tema, dados_tema in list(temas.items()):
#         novos_subs = []
#         for item in dados_tema.get("subtemas_vinculados", []):
#             item_clean = item.upper().strip()
#             if item_clean not in subtemas:
#                 sub_verdadeiro = None
#                 for sub_k, sub_v in subtemas.items():
#                     if item_clean in [c.upper().strip() for c in sub_v.get("codigos_vinculados", [])]:
#                         sub_verdadeiro = sub_k
#                         break
#                 if sub_verdadeiro:
#                     if sub_verdadeiro not in novos_subs: novos_subs.append(sub_verdadeiro)
#                     print(f"  -> Correção Esquema: Código '{item_clean}' removido de '{tema}' e remapeado para o Subtema mestre '{sub_verdadeiro}'.")
#             else:
#                 if item_clean not in novos_subs: novos_subs.append(item_clean)
#         dados_tema["subtemas_vinculados"] = novos_subs

#     # 2. Unifica e consolida homônimos suffixados por colisões anteriores
#     def consolidar_sufixos(dic_mestre, tipo_nome):
#         for k in list(dic_mestre.keys()):
#             if k.endswith("_") and k in dic_mestre:
#                 k_limpo = re.sub(r'_+$', '', k).upper().strip()
#                 if k_limpo in dic_mestre and k_limpo != k:
#                     print(f"  -> Fusão de Homônimos: Unificando '{k}' in '{k_limpo}' ({tipo_nome}).")
#                     chave_v = "codigos_vinculados" if tipo_nome == "SUBTEMA" else "subtemas_vinculados"
#                     for v in dic_mestre[k].get(chave_v, []):
#                         if v not in dic_mestre[k_limpo][chave_v]: dic_mestre[k_limpo][chave_v].append(v)
#                     dic_mestre[k_limpo]["justificativas_agrupamento"].extend(dic_mestre[k].get("justificativas_agrupamento", []))
#                     del dic_mestre[k]

#     consolidar_sufixos(subtemas, "SUBTEMA")
#     consolidar_sufixos(temas, "TEMA")

#     # Sincroniza ponteiros de amarração
#     for tema, dados_tema in temas.items():
#         dados_tema["subtemas_vinculados"] = [re.sub(r'_+$', '', s).upper().strip() for s in dados_tema.get("subtemas_vinculados", [])]

#     with open(ARQUIVO_CODEBOOK_TEMAS, 'w', encoding='utf-8') as f: json.dump(temas, f, ensure_ascii=False, indent=4)
#     with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'w', encoding='utf-8') as f: json.dump(subtemas, f, ensure_ascii=False, indent=4)
#     print("[SUCESSO] Snapshot de dados limpo e estruturado em disco.")

# reparar_dados_corrompidos_offline()